# BdSL Gloss-to-Pose

Gloss-to-pose generation for Bangladeshi Sign Language (BdSL): a Bangla gloss sentence is encoded
with BanglaBERT and mapped to a sequence of 95 body, hand and head landmarks. This notebook is
generated by `scripts/build_notebook.py`; the repository README gives the research context, the
results, and how each experiment variant is built.

Dataset: **1,922 sentences x 5 signers = 9,610 recordings**, laid out as
`TENSOR_DIR/<sentence_id>/*.npy` with five `.npy` files per sentence folder.

---

## What the notebook contains

- Three model families trained and evaluated under one protocol: a diffusion generator
  (NAT + Diffusion + Velocity), a non-autoregressive regressor (NAT + MSE), and an autoregressive
  Progressive Transformer.
- DTW-aligned evaluation: DTW-MJE, multi-threshold PCK, length ratio, motion ratio, jitter ratio.
- Optional studies switched on by build flags: classifier-free-guidance sweep, test-time
  smoothness guidance, a temporal-variance penalty for the NAT model, and back-translation.

## Experimental design

- **Splits.** 80/10/10 by normalised gloss text (`DEDUP_BY_GLOSS`), not by recording or sentence
  ID: five signers record each sentence and gloss strings repeat across IDs, so a recording- or
  ID-level split leaks identical sentences into the test set. The partition is chosen before any
  signer filtering, so single-signer and all-signer runs hold out the same sentences. An optional
  signer-held-out mode trains on four signers and tests on the fifth.
- **Single signer by default** (`SIGNERS_TO_USE = [0]`): extra signers add pose diversity, not
  vocabulary, and a deployed system renders one avatar. The other signers are used for the
  cross-signer evaluation. An optional signer embedding supports multi-signer training.
- **Pose normalisation.** Each sequence is centred on its median mid-shoulder point and scaled by
  its median shoulder width, so all errors are in shoulder-width units.
- **Objective.** Padding is masked everywhere; Soft-DTW is divided by sequence length; landmarks
  23-32 (extrapolated by the pose extractor) are down-weighted; optional velocity and bone-length
  terms.
- **Optimisation.** Separate learning rates for BanglaBERT and new layers, warmup + cosine,
  gradient clipping, mixed precision with the loss in fp32, per-epoch validation, early stopping
  on an EMA of validation loss, best/last checkpoints, full seeding.
- **Evaluation.** Error is measured along a DTW alignment path, so duration error is not charged
  to the pose model.


# **1. Installation**

In [ ]:
!pip install transformers tslearn -q


# **2. Configuration & Seeding**

Everything tunable lives here. `SPLIT_MODE` is the switch between the main experiment (`"sentence"`) and the signer-generalization experiment
(`"signer_holdout"`).

In [ ]:
import os
import random
import numpy as np
import torch


class Config:
    # ---------------- Paths: edit for your environment ----------------
    # (a path that does not exist is searched for by file name under /kaggle/input)
    EXCEL_PATH = "/kaggle/input/bdsl-sentences/Bangla_Sign_Sentence.xlsx"
    TENSOR_DIR = "/kaggle/input/bdsl-pose-tensors/BdSL_Pose_Tensors"
    SHEET_NAME = "Training Data"
    REF_IMAGE_PATH = "/kaggle/input/bdsl-reference-image/reference_signer.png"
    CHECKPOINT_DIR = "/kaggle/working"
    # A "Save & Run All" commit does NOT inherit this kernel's own previous
    # /kaggle/working across versions -- each run starts clean. To resume from a prior
    # run, upload its g2p_last.pt / g2p_progressive_last.pt / g2p_nat_mse_last.pt
    # as a Kaggle dataset, attach it, and point this at its mount path so
    # RESUME=True (below) actually has something to resume from, instead of
    # silently retraining every model from scratch. None = no staging.
    RESUME_SOURCE_DIR = None
    # None = copy every file in RESUME_SOURCE_DIR. A list = copy only these file
    # names, each located by recursive search under RESUME_SOURCE_DIR (needed
    # when the source is a kernel's output, whose mount path is not fixed).
    RESUME_SOURCE_FILES = None
    # False in the velocity/guidance ablation notebooks, which train and
    # evaluate only the diffusion model.
    BUILD_COMPARISON_MODELS = True
    # Section 16e: re-score the best checkpoint at several classifier-free
    # guidance scales. Guidance acts only at sampling time, so this costs a few
    # minutes of evaluation, not a training run.
    RUN_CFG_SWEEP = False
    # Section 16f: test-time smoothness guidance on the diffusion sampler, with
    # plain post-hoc smoothing as the control. Inference-only.
    RUN_SMOOTH_GUIDANCE = False
    # Weight of the temporal-variance matching term in the NAT+MSE variant
    # (Section 9b/10b). 0.0 = plain MSE, i.e. exactly the plain NAT + MSE model.
    VARIANCE_WEIGHT = 0.0
    # Which models Sections 14/14b train. The NAT-variance study turns the first
    # two off so only the NAT variant (14c) trains.
    TRAIN_DIFFUSION = True
    TRAIN_PROGRESSIVE = True
    # Section 16g: evaluate the NAT variant (val / test / cross-signer).
    RUN_NAT_EVAL = False
    # Sections 16h-16j: back-translation. A CTC recogniser (pose -> gloss tokens) is trained
    # on REAL training poses only, then reads poses generated by each model; accuracy
    # against the input gloss measures whether the signing carries the meaning.
    RUN_BACKTRANS = False
    REC_EPOCHS = 80
    REC_BATCH = 32
    REC_LR = 3e-4
    REC_PATIENCE = 15
    REC_BOOTSTRAP = 2000
    BT_CFG_SCALES = (1.0, 2.5, 4.0)
    CFG_SWEEP = (1.0, 1.5, 2.5, 4.0)

    # ---------------- Reproducibility ----------------
    SEED = 42

    # ---------------- Split design ----------------
    # Sentence IDs are NOT unique gloss strings: the corpus is combinatorial
    # ({kinship} x {time} x {food} x {polarity}) and the same gloss text recurs
    # under different IDs. Measured on the ID-disjoint split, 15.1% of test
    # sentences had a training gloss at cosine similarity >= 0.95 and four were
    # verbatim identical. With this True, the partition is by normalised gloss
    # TEXT, so an identical sentence can never straddle the split.
    DEDUP_BY_GLOSS = True
    # "sentence"       -> sentence-disjoint 80/10/10 (MAIN EXPERIMENT)
    # "signer_holdout" -> train on 4 signers, test on the unseen 5th
    #                     (sentences also disjoint, so it is a strict test)
    SPLIT_MODE = "sentence"
    VAL_FRAC = 0.10
    TEST_FRAC = 0.10
    HELDOUT_SIGNER = 4  # only used when SPLIT_MODE == "signer_holdout"

    # ---------------- Which signers to train on ----------------
    # [0]           -> SINGLE-SIGNER: 1,923 videos, ~1.7 h per 200-epoch run.
    #                  Recommended for development and for the ablation table.
    # None          -> ALL 5 signers: 9,610 videos, ~8.3 h per run (AMP on).
    # [0, 1]        -> any subset also works.
    # Linguistic coverage is 1,923 sentences either way — extra signers add pose
    # diversity, not new vocabulary.
    SIGNERS_TO_USE = [0]

    # Evaluate on signers the model never trained on (costs NO extra training).
    # Requires SIGNERS_TO_USE to be a strict subset of all signers.
    CROSS_SIGNER_EVAL = True

    # ---------------- Text encoder ----------------
    BANGLABERT_MODEL = "csebuetnlp/banglabert"
    MAX_TEXT_LEN = 16
    # False = fine-tune all ~110M BanglaBERT params. Early stopping guards the
    # overfitting risk, but with only ~1,539 training sentences, True (frozen,
    # train the projection head only) is a cheap and worthwhile A/B — run it
    # second and keep whichever wins on validation.
    FREEZE_BANGLABERT = False

    # ---------------- Model dims ----------------
    D_MODEL = 256
    NUM_JOINTS = 95
    COORD_DIM = 3
    POSE_DIM = NUM_JOINTS * COORD_DIM  # 285

    # ---------------- Signer conditioning ----------------
    # With SIGNERS_TO_USE=[0] this degenerates to a constant bias (one embedding
    # row, added to every sample) so it is off. Turn it ON only for a
    # multi-signer run, where it is what stops the model averaging five people.
    USE_SIGNER_EMBEDDING = False
    NUM_SIGNERS = 5

    # ---------------- Pose normalization ----------------
    NORMALIZE_POSES = True        # set False for the ablation
    MIN_SHOULDER_SCALE = 1e-3     # guard against degenerate detections

    # ---------------- Diffusion ----------------
    DIFFUSION_TIMESTEPS = 250
    SAMPLING_STEPS = 50

    # ---------------- Training ----------------
    BATCH_SIZE = 16               # raised from 8: better GPU utilization on T4
    NUM_WORKERS = 2
    LR = 2e-4                     # fresh layers (scaled with the larger batch)
    BERT_LR = 3e-5                # pretrained BanglaBERT
    WEIGHT_DECAY = 1e-4
    # ~96 batches/epoch at single-signer, so an epoch is seconds, not minutes.
    # Early stopping ends the run; a high ceiling just avoids under-training.
    EPOCHS = 500
    WARMUP_EPOCHS = 10
    GRAD_CLIP_NORM = 1.0
    USE_AMP = True

    # Group same-length videos into the same batch. Random batching wastes
    # ~35-40% of compute on padding because a 30-frame and a 300-frame video
    # in one batch both pad to 300.
    LENGTH_GROUPED_BATCHING = True

    # Kaggle kills the session at 12 h. Stop cleanly before that so the run
    # always ends with a usable checkpoint instead of being cut mid-epoch.
    TIME_BUDGET_HOURS = 10.5
    # Budget for EACH of the two comparison models in Sections 14b/14c, separate
    # from the diffusion model's budget above. At SIGNERS_TO_USE=[0] a 200-epoch run is ~1.7-2.5h,
    # so training both comparison models after the diffusion model still fits inside one
    # Kaggle session. Lower this (or train them in a separate session) if
    # SIGNERS_TO_USE is None (all 5 signers).
    COMPARISON_TIME_BUDGET_HOURS = 3.5
    # Set True to continue an interrupted run from g2p_last.pt.
    RESUME = True
    # Full resume state is ~1.4 GB; writing it every epoch would burn ~30 min of
    # pure disk I/O over a 200-epoch run. Every 5 epochs costs at most 5 epochs
    # of lost progress on a crash.
    SAVE_LAST_EVERY = 5

    # Val loss is noisy (std ~0.3), so one lucky epoch can set a bar the run
    # never beats — an early run stopped at epoch 149 while still improving. Early
    # stopping now tracks an EMA of val loss.
    # Raised from 60 -> 150 after the 2026-10-01/02 architecture-comparison run:
    # the diffusion model early-stopped at epoch 210 (best raw val at 114) while
    # the other two models trained to 500/407 epochs, and the diffusion model
    # lost on DTW-MJE/PCK to both of them. Before trusting that comparison, find out
    # whether 60-epoch patience cut the diffusion model off prematurely on its
    # documented-noisy val loss, or whether it had genuinely plateaued.
    EARLY_STOP_PATIENCE = 150
    VAL_EMA = 0.8
    CHECKPOINT_EVERY = 50
    EVAL_EVERY = 50               # epochs between (expensive) generation evals
    EVAL_SUBSET_BATCHES = 8       # batches used for the periodic generation eval

    MAX_GEN_FRAMES = 300          # safety cap at inference

    # ---------------- Loss weights ----------------
    BONE_WEIGHT = 20.0            # was 1.5: measured at only 0.45% of the loss
    HAND_WEIGHT = 1.5
    DURATION_WEIGHT = 1.0
    SOFT_DTW_GAMMA = 0.1

    # Soft-DTW does NOT penalise a frozen output: DTW maps many target frames
    # onto one predicted frame at no cost, so a static mean pose is a valid
    # minimiser. This term scores frame-to-frame DELTAS, which a static output
    # fails outright. Added after an early run collapsed to a static output; a later ablation of this weight is in docs/experiments/.
    VELOCITY_WEIGHT = 10.0

    # Classifier-free guidance: drop the text condition sometimes during
    # training, then at sampling push away from the unconditional prediction.
    # Amplifies how strongly the gloss steers the output.
    CFG_DROPOUT = 0.1             # P(drop condition) while training
    CFG_SCALE = 2.5               # 1.0 disables guidance at sampling time

    # ---------------- Skeleton indices (MediaPipe Holistic) ----------------
    SHOULDER_L, SHOULDER_R = 11, 12
    ELBOW_L, ELBOW_R = 13, 14
    WRIST_L, WRIST_R = 15, 16
    HIP_L, HIP_R = 23, 24
    KNEE_L, KNEE_R = 25, 26
    ANKLE_L, ANKLE_R = 27, 28
    HAND_START, HAND_END = 33, 75   # 21 left + 21 right
    # Landmarks 75..94 are whatever extra points the extractor kept
    # (commonly a face subset). They are modeled but not used structurally.

    # Landmarks 23..32 (hips/knees/ankles/heels/feet) are EXTRAPOLATED by the
    # pose extractor, not observed: left/right medians agree to 0.01, the
    # shoulder-to-hip distance is 2.40 shoulder-widths where anatomy says ~1.25,
    # and they carry the largest coordinate magnitudes in the dataset (up to
    # 7.5). Because Soft-DTW runs on the flattened coordinate vector, their
    # variance dominates the objective. Downweighting them hands the loss back
    # to the hands, which is where a sign's meaning actually lives.
    LEG_START, LEG_END = 23, 33
    LEG_WEIGHT = 0.1                # set 1.0 for the ablation row

    BONE_PAIRS = [
        # Upper body + torso only. Knee/ankle bones are dropped: constraining a
        # bone between two invented landmarks just injects the extractor's
        # extrapolation noise into the gradient. Hip bones are kept because the
        # loss anchors each bone to that sample's own mean length, so a
        # consistently-placed hip is still a usable torso reference.
        (SHOULDER_L, SHOULDER_R), (SHOULDER_L, ELBOW_L), (ELBOW_L, WRIST_L),
        (SHOULDER_R, ELBOW_R), (ELBOW_R, WRIST_R),
        (SHOULDER_L, HIP_L), (SHOULDER_R, HIP_R), (HIP_L, HIP_R),
    ]


cfg = Config()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(cfg.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Seed:   {cfg.SEED}")
print(f"Split:  {cfg.SPLIT_MODE}")
print(f"Signer embedding: {cfg.USE_SIGNER_EMBEDDING} | Pose normalization: {cfg.NORMALIZE_POSES}")


# **2b. Resume Staging (optional)**

A "Save & Run All" commit does not inherit this kernel's own `/kaggle/working`
from a previous version — each run starts clean. If `RESUME_SOURCE_DIR` points
at an attached dataset holding a prior run's `*_last.pt` / `len_cache.json`,
this copies them into `CHECKPOINT_DIR` before training starts, so `RESUME=True`
in Sections 14/14b/14c has something real to resume from instead of silently
retraining every model from scratch.

In [ ]:
import glob as _glob
import shutil

if cfg.RESUME_SOURCE_DIR and os.path.isdir(cfg.RESUME_SOURCE_DIR):
    staged = 0
    if cfg.RESUME_SOURCE_FILES is None:
        to_copy = [(f, os.path.join(cfg.RESUME_SOURCE_DIR, f))
                   for f in os.listdir(cfg.RESUME_SOURCE_DIR)]
    else:
        to_copy = []
        for want in cfg.RESUME_SOURCE_FILES:
            hits = _glob.glob(os.path.join(cfg.RESUME_SOURCE_DIR, "**", want), recursive=True)
            if hits:
                to_copy.append((want, hits[0]))
            else:
                print(f"WARNING: {want} not found under {cfg.RESUME_SOURCE_DIR}")
    for fname, src in to_copy:
        dst = os.path.join(cfg.CHECKPOINT_DIR, fname)
        if os.path.isfile(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
            staged += 1
            print(f"staged {fname} ({os.path.getsize(src)/1e6:.0f} MB) -> {dst}")
    print(f"\nResume staging: {staged} file(s) copied from {cfg.RESUME_SOURCE_DIR}")
else:
    print(f"No resume source at {cfg.RESUME_SOURCE_DIR!r}; any model without an "
         "existing checkpoint in CHECKPOINT_DIR will train from scratch.")

# Attaching a kernel as an input (kernel_sources) changes the /kaggle/input
# layout, so the hardcoded dataset paths above can stop existing. If a path is
# missing, locate the data by name instead of failing deep inside the loader.
def _find_input(pattern, want_dir):
    hits = [h for h in _glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True)
            if os.path.isdir(h) == want_dir]
    return hits[0] if hits else None

for _attr, _pat, _dir in [("EXCEL_PATH", "Bangla_Sign_Sentence.xlsx", False),
                          ("TENSOR_DIR", "BdSL_Pose_Tensors", True),
                          ("REF_IMAGE_PATH", "reference_signer.png", False)]:
    if not os.path.exists(getattr(cfg, _attr)) and os.path.isdir("/kaggle/input"):
        _hit = _find_input(_pat, _dir)
        print(f"{_attr} {getattr(cfg, _attr)!r} not found -> "
              f"{'using ' + repr(_hit) if _hit else 'NOT FOUND under /kaggle/input'}")
        if _hit:
            setattr(cfg, _attr, _hit)


# **3. Dataset, Normalization & Splits**

**Signer identity** is parsed from the filename token `_U<number>_` (not from sorted position,
which orders U19 before U4) and mapped to indices 0..N-1 by numeric order. A diagnostic prints
sentence and signer counts so the file naming can be checked.

**Poses are normalized per sequence.** Centre = median mid-shoulder point over the sequence;
scale = median shoulder width. Using medians over frames rather than per-frame values removes
signer position and body size while preserving global body movement. Centre and scale are
returned with each sample so rendering can invert the transform.

**Splits are by normalised gloss text, never by recording.** Five signers record every sentence
and the corpus is combinatorial, so identical gloss strings recur under different sentence IDs.
On an ID-disjoint split, 15.1% of test sentences had a near-duplicate training gloss.
`DEDUP_BY_GLOSS` partitions on the normalised gloss text; assertions verify that no sentence or
gloss appears in more than one split.


In [ ]:
import glob
import json
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer


import re as _re

# Signer identity is encoded explicitly in the filename as _U<number>_
#   UL_CS_S23511_U19_20231010_1696931429197.npy  ->  user 19
# Parsing that is strictly safer than sorted-filename position: string sort puts
# U19 before U4, and any folder holding a different set of users would silently
# shift every position. Users are mapped to 0..N-1 by numeric order, so
# signer_id 0 = U4 (the lowest user number), NOT U19 as positional order gave.
SIGNER_RE = _re.compile(r'_U(\d+)_')
SIGNER_LUT = {}   # populated in build_records


def parse_user_number(npy_path):
    m = SIGNER_RE.search(os.path.basename(npy_path))
    return int(m.group(1)) if m else None


def extract_signer_id(npy_path, position_in_folder):
    """Stable signer index from the filename's _U<n>_ token.

    Falls back to sorted position only if the filename cannot be parsed, so a
    dataset with a different naming scheme still works.
    """
    u = parse_user_number(npy_path)
    if u is None or u not in SIGNER_LUT:
        return position_in_folder
    return SIGNER_LUT[u]


class BdSLGlossToPoseDataset(Dataset):
    def __init__(self, records, tokenizer, max_seq_len=cfg.MAX_TEXT_LEN,
                 normalize=cfg.NORMALIZE_POSES):
        self.records = records
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.normalize = normalize

    def __len__(self):
        return len(self.records)

    def _normalize_pose(self, pose):
        """pose: (T, J, 3) tensor -> normalized pose, center (3,), scale scalar."""
        shoulders = pose[:, [cfg.SHOULDER_L, cfg.SHOULDER_R], :]     # (T, 2, 3)
        center = shoulders.mean(dim=1)                               # (T, 3)
        center = center.median(dim=0).values                         # (3,)

        widths = torch.norm(
            pose[:, cfg.SHOULDER_L, :] - pose[:, cfg.SHOULDER_R, :], dim=-1
        )                                                            # (T,)
        scale = widths.median()
        scale = torch.clamp(scale, min=cfg.MIN_SHOULDER_SCALE)

        normed = (pose - center.view(1, 1, 3)) / scale
        return normed, center, scale

    def __getitem__(self, idx):
        rec = self.records[idx]

        encoded = self.tokenizer(
            rec['gloss'],
            padding='max_length',
            max_length=self.max_seq_len,
            truncation=True,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)

        pose = torch.tensor(np.load(rec['tensor_path']), dtype=torch.float32)
        if pose.ndim == 2:  # (T, 285) -> (T, 95, 3)
            pose = pose.view(pose.shape[0], cfg.NUM_JOINTS, cfg.COORD_DIM)

        if self.normalize:
            pose, center, scale = self._normalize_pose(pose)
        else:
            center = torch.zeros(3)
            scale = torch.tensor(1.0)

        signer_id = torch.tensor(rec['signer_id'], dtype=torch.long)
        return input_ids, attention_mask, pose, signer_id, center, scale


def pad_collate_fn(batch):
    input_ids = torch.stack([b[0] for b in batch])
    attention_mask = torch.stack([b[1] for b in batch])
    poses = [b[2] for b in batch]
    signer_ids = torch.stack([b[3] for b in batch])
    centers = torch.stack([b[4] for b in batch])
    scales = torch.stack([b[5] for b in batch])

    padded = pad_sequence(poses, batch_first=True, padding_value=0.0)
    lengths = torch.tensor([p.shape[0] for p in poses], dtype=torch.long)
    return input_ids, attention_mask, padded, lengths, signer_ids, centers, scales


def build_records(excel_path, tensor_dir, sheet_name):
    df = pd.read_excel(excel_path, sheet_name=sheet_name)
    df['Sentence ID'] = df['Sentence ID'].astype(str)

    # Pass 1: collect every user number so ids are assigned by numeric order.
    users = set()
    for _, row in df.iterrows():
        folder = os.path.join(tensor_dir, str(row['Sentence ID']))
        if os.path.exists(folder):
            for p in glob.glob(os.path.join(folder, "*.npy")):
                u = parse_user_number(p)
                if u is not None:
                    users.add(u)
    SIGNER_LUT.clear()
    SIGNER_LUT.update({u: i for i, u in enumerate(sorted(users))})
    if SIGNER_LUT:
        print("Signer map (filename user -> signer_id): "
              + ", ".join(f"U{u}->{i}" for u, i in SIGNER_LUT.items()))
    else:
        print("No _U<n>_ tokens found in filenames; falling back to sorted position.")

    records, per_sentence_counts, sample_names = [], {}, []
    for _, row in df.iterrows():
        sent_id = row['Sentence ID']
        gloss = str(row['Sign Sentence'])
        folder = os.path.join(tensor_dir, sent_id)
        if not os.path.exists(folder):
            continue
        npys = sorted(glob.glob(os.path.join(folder, "*.npy")))
        if not npys:
            continue
        per_sentence_counts[sent_id] = len(npys)
        if len(sample_names) < 3:
            sample_names.append((sent_id, [os.path.basename(p) for p in npys]))
        for pos, path in enumerate(npys):
            records.append({
                'sentence_id': sent_id,
                'gloss': gloss,
                'tensor_path': path,
                'signer_id': extract_signer_id(path, pos),
            })

    if not records:
        raise RuntimeError("No (gloss, pose) pairs found. Check EXCEL_PATH / TENSOR_DIR.")
    return records, per_sentence_counts, sample_names


def make_splits(records, mode, seed):
    """Sentence-disjoint splitting. All signers of a sentence stay together.

    Returns (train, val, test, cross_signer_test). The sentence-level partition is
    decided BEFORE any signer filtering, so the held-out sentences are identical no
    matter which signers are used for training — that keeps single-signer and all-signer runs
    directly comparable.
    """
    def _key(r):
        if not cfg.DEDUP_BY_GLOSS:
            return r['sentence_id']
        # normalise: collapse whitespace, drop the sentence-final danda
        g = " ".join(str(r['gloss']).split()).strip().rstrip("।.")
        return "GLOSS::" + g

    for r in records:
        r['split_key'] = _key(r)

    keys = sorted({r['split_key'] for r in records})
    rng = random.Random(seed)
    rng.shuffle(keys)

    n = len(keys)
    n_test = int(round(n * cfg.TEST_FRAC))
    n_val = int(round(n * cfg.VAL_FRAC))
    test_ids = set(keys[:n_test])
    val_ids = set(keys[n_test:n_test + n_val])
    train_ids = set(keys[n_test + n_val:])

    train = [r for r in records if r['split_key'] in train_ids]
    val = [r for r in records if r['split_key'] in val_ids]
    test = [r for r in records if r['split_key'] in test_ids]

    if mode == "signer_holdout":
        train = [r for r in train if r['signer_id'] != cfg.HELDOUT_SIGNER]
        val = [r for r in val if r['signer_id'] != cfg.HELDOUT_SIGNER]
        test = [r for r in test if r['signer_id'] == cfg.HELDOUT_SIGNER]
        return train, val, test, []

    keep = cfg.SIGNERS_TO_USE
    cross = []
    if keep is not None:
        keep = set(keep)
        # Unseen signers on unseen sentences -> free generalization test set.
        cross = [r for r in test if r['signer_id'] not in keep]
        train = [r for r in train if r['signer_id'] in keep]
        val = [r for r in val if r['signer_id'] in keep]
        test = [r for r in test if r['signer_id'] in keep]

    return train, val, test, cross


print("Indexing dataset...")
all_records, counts, samples = build_records(cfg.EXCEL_PATH, cfg.TENSOR_DIR, cfg.SHEET_NAME)

n_sentences = len(counts)
videos_per_sentence = {}
for c in counts.values():
    videos_per_sentence[c] = videos_per_sentence.get(c, 0) + 1

print(f"\nSentences with data : {n_sentences}")
print(f"Total videos        : {len(all_records)}")
print(f"Videos per sentence : {videos_per_sentence}   (expect mostly {{5: ...}})")
print("\nFilename check (first 3 sentence folders) — confirm naming is consistent:")
for sid, names in samples:
    print(f"  {sid}: {names}")

signer_hist = {}
for r in all_records:
    signer_hist[r['signer_id']] = signer_hist.get(r['signer_id'], 0) + 1
print(f"\nSigner distribution: {dict(sorted(signer_hist.items()))}")

tokenizer = AutoTokenizer.from_pretrained(cfg.BANGLABERT_MODEL)
train_recs, val_recs, test_recs, cross_recs = make_splits(all_records, cfg.SPLIT_MODE, cfg.SEED)

# Hard assertion: no sentence may appear in more than one split.
tr_s = {r['sentence_id'] for r in train_recs}
va_s = {r['sentence_id'] for r in val_recs}
te_s = {r['sentence_id'] for r in test_recs}
assert not (tr_s & va_s), "LEAK: sentence in both train and val"
assert not (tr_s & te_s), "LEAK: sentence in both train and test"
assert not (va_s & te_s), "LEAK: sentence in both val and test"

TRAIN_SIGNERS = sorted({r['signer_id'] for r in train_recs})
PRIMARY_SIGNER = TRAIN_SIGNERS[0] if TRAIN_SIGNERS else 0

print(f"\n--- Split ({cfg.SPLIT_MODE}) ---")
print(f"Train: {len(train_recs):5d} videos / {len(tr_s):4d} sentences  signers={TRAIN_SIGNERS}")
print(f"Val  : {len(val_recs):5d} videos / {len(va_s):4d} sentences")
print(f"Test : {len(test_recs):5d} videos / {len(te_s):4d} sentences")
if cross_recs:
    cs = sorted({r['signer_id'] for r in cross_recs})
    print(f"Cross-signer test: {len(cross_recs):5d} videos, signers={cs} "
          f"(unseen signers on the SAME unseen sentences)")
unit = "unique glosses" if cfg.DEDUP_BY_GLOSS else "sentence IDs"
print(f"Disjointness verified over {unit}.")
n_ids = len({r['sentence_id'] for r in all_records})
n_gl = len({" ".join(str(r['gloss']).split()).strip().rstrip("।.") for r in all_records})
print(f"\n{n_ids} sentence IDs map to {n_gl} unique gloss strings "
      f"({n_ids - n_gl} duplicate IDs, {100*(n_ids-n_gl)/max(n_ids,1):.1f}%).")
if not cfg.DEDUP_BY_GLOSS and n_gl < n_ids:
    print("WARNING: duplicate gloss strings exist and DEDUP_BY_GLOSS is False, so "
          "the same sentence text can appear in both train and test.")

if len(TRAIN_SIGNERS) < 2 and cfg.USE_SIGNER_EMBEDDING:
    print("\n[NOTE] Training on a single signer, so the signer embedding degenerates to a "
          "constant bias — harmless but it does nothing. Set USE_SIGNER_EMBEDDING=False "
          "for a cleaner single-signer model.")

train_ds = BdSLGlossToPoseDataset(train_recs, tokenizer)
val_ds = BdSLGlossToPoseDataset(val_recs, tokenizer)
test_ds = BdSLGlossToPoseDataset(test_recs, tokenizer)
cross_ds = BdSLGlossToPoseDataset(cross_recs, tokenizer) if cross_recs else None


def _worker_init(worker_id):
    np.random.seed(cfg.SEED + worker_id)
    random.seed(cfg.SEED + worker_id)


_g = torch.Generator()
_g.manual_seed(cfg.SEED)


LENGTH_CACHE = "len_cache.json"


def scan_lengths(dataset, cache_name=None):
    """Frame count per sample, read from .npy headers (no full load).

    One shared cache keyed by absolute path, so it stays valid when the split
    changes. Only files not already in the cache are scanned. Matching on
    membership rather than count matters: changing SIGNERS_TO_USE, or switching
    signer ids from sorted-position to filename-parsed, swaps *which* files a
    split holds while leaving the count identical.
    """
    cache = os.path.join(cfg.CHECKPOINT_DIR, LENGTH_CACHE)
    paths = [r['tensor_path'] for r in dataset.records]

    lens = {}
    if os.path.exists(cache):
        try:
            lens = json.load(open(cache))
        except Exception:
            lens = {}   # corrupt or truncated cache: just rebuild it

    missing = [p for p in paths if p not in lens]
    if missing:
        for p in tqdm(missing, desc=f"Scanning lengths ({len(missing)} new)"):
            lens[p] = int(np.load(p, mmap_mode='r').shape[0])
        json.dump(lens, open(cache, "w"))

    return [lens[p] for p in paths]


class LengthGroupedBatchSampler(torch.utils.data.Sampler):
    """Batches of similarly-long sequences, so padding barely costs anything.

    Shuffled mode (training): shuffle, cut into mega-batches, sort each
    mega-batch by length, emit batches, then shuffle the batch order. Keeps
    epoch-to-epoch randomness while keeping each batch length-homogeneous.
    Sorted mode (val/test): globally sorted, deterministic.
    """

    def __init__(self, lengths, batch_size, shuffle=True, seed=0,
                 drop_last=False, mega_batch_mult=50):
        self.lengths = list(lengths)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.drop_last = drop_last
        self.mega = batch_size * mega_batch_mult
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def _batches(self):
        n = len(self.lengths)
        if self.shuffle:
            g = random.Random(self.seed + self.epoch)
            idx = list(range(n))
            g.shuffle(idx)
        else:
            idx = sorted(range(n), key=lambda i: self.lengths[i])

        batches = []
        for s in range(0, n, self.mega):
            chunk = idx[s:s + self.mega]
            if self.shuffle:
                chunk.sort(key=lambda i: self.lengths[i])
            for b in range(0, len(chunk), self.batch_size):
                batch = chunk[b:b + self.batch_size]
                if self.drop_last and len(batch) < self.batch_size:
                    continue
                batches.append(batch)
        if self.shuffle:
            random.Random(self.seed + 10007 + self.epoch).shuffle(batches)
        return batches

    def __iter__(self):
        return iter(self._batches())

    def __len__(self):
        return len(self._batches())


def make_loader(ds, shuffle, drop_last=False):
    if ds is None:
        return None, None
    if not cfg.LENGTH_GROUPED_BATCHING:
        return DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=shuffle,
                          collate_fn=pad_collate_fn, num_workers=cfg.NUM_WORKERS,
                          pin_memory=True, worker_init_fn=_worker_init,
                          generator=_g, drop_last=drop_last), None
    lens = scan_lengths(ds)
    sampler = LengthGroupedBatchSampler(lens, cfg.BATCH_SIZE, shuffle=shuffle,
                                        seed=cfg.SEED, drop_last=drop_last)
    loader = DataLoader(ds, batch_sampler=sampler, collate_fn=pad_collate_fn,
                        num_workers=cfg.NUM_WORKERS, pin_memory=True,
                        worker_init_fn=_worker_init)
    return loader, sampler


train_loader, train_sampler = make_loader(train_ds, True, drop_last=True)
val_loader, _ = make_loader(val_ds, False)
test_loader, _ = make_loader(test_ds, False)
cross_loader, _ = make_loader(cross_ds, False)

print(f"\nBatches -> train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)}"
      + (f" | cross-signer {len(cross_loader)}" if cross_loader else ""))

if cfg.LENGTH_GROUPED_BATCHING:
    tl = scan_lengths(train_ds)
    rnd = random.Random(0); shuf = list(tl); rnd.shuffle(shuf)
    def _pad_frac(order):
        real = pad = 0
        for i in range(0, len(order) - cfg.BATCH_SIZE + 1, cfg.BATCH_SIZE):
            b = order[i:i + cfg.BATCH_SIZE]
            real += sum(b); pad += max(b) * len(b)
        return 1 - real / max(pad, 1)
    print(f"\nPadding overhead: {100*_pad_frac(shuf):.1f}% random order  ->  "
          f"{100*_pad_frac(sorted(tl)):.1f}% length-grouped")


# **4. Dataset Statistics**

Sequence-length distribution (to check `MAX_GEN_FRAMES`) and a sanity check on normalization:
shoulder width is 1.0 by construction, so normalized coordinates should mostly fall within about
-3 to 3.


In [ ]:
from tqdm import tqdm

lengths = []
sample_n = min(400, len(train_ds))
for i in tqdm(range(sample_n), desc="Scanning sequence lengths"):
    p = np.load(train_ds.records[i]['tensor_path'], mmap_mode='r')
    lengths.append(p.shape[0])

lengths = np.array(lengths)
print("\n--- Sequence length (frames) ---")
print(f"count : {len(lengths)}")
print(f"min   : {lengths.min()}")
print(f"max   : {lengths.max()}")
print(f"mean  : {lengths.mean():.1f}")
print(f"median: {np.median(lengths):.1f}")
print(f"p95   : {np.percentile(lengths, 95):.1f}")
print(f"p99   : {np.percentile(lengths, 99):.1f}")
print(f"\nMAX_GEN_FRAMES is {cfg.MAX_GEN_FRAMES} — should comfortably exceed p99.")

ii, am, pose, sid, center, scale = train_ds[0]
print(f"\n--- Sample 0 (signer {sid.item()}) ---")
print(f"pose shape        : {tuple(pose.shape)}")
print(f"normalized ranges : x[{pose[...,0].min():.2f}, {pose[...,0].max():.2f}] "
      f"y[{pose[...,1].min():.2f}, {pose[...,1].max():.2f}] "
      f"z[{pose[...,2].min():.2f}, {pose[...,2].max():.2f}]")
print(f"denorm center     : {center.numpy().round(3)}")
print(f"denorm scale      : {scale.item():.4f}")
sw = torch.norm(pose[:, cfg.SHOULDER_L] - pose[:, cfg.SHOULDER_R], dim=-1).median()
print(f"median shoulder width after normalization: {sw.item():.4f}  (should be ~1.0)")


# **4b. Data Quality Check (run this BEFORE any long training)**

Silent data corruption is the cheapest way to waste an 8-hour run. NaNs
propagate through the loss and turn every gradient to NaN; all-zero frames from
failed pose detection teach the model to predict nothing; near-zero shoulder
widths blow up the normalization divisor.

This samples a few hundred sequences and reports all three. Run it once before a long run; it takes about a minute.

In [ ]:
def quality_check(dataset, n_samples=300, seed=cfg.SEED):
    rng = random.Random(seed)
    idxs = rng.sample(range(len(dataset)), min(n_samples, len(dataset)))

    n_nan = n_inf = n_allzero_seq = 0
    zero_frame_frac, tiny_scale, jumps = [], 0, []
    shoulder_widths = []

    for i in tqdm(idxs, desc="Quality check"):
        rec = dataset.records[i]
        p = np.load(rec['tensor_path'])
        if p.ndim == 2:
            p = p.reshape(p.shape[0], cfg.NUM_JOINTS, cfg.COORD_DIM)

        if np.isnan(p).any():
            n_nan += 1
        if np.isinf(p).any():
            n_inf += 1
        if not np.any(p):
            n_allzero_seq += 1
            continue

        # fraction of frames where every landmark is exactly zero (detection dropout)
        zf = (np.abs(p).sum(axis=(1, 2)) == 0).mean()
        zero_frame_frac.append(zf)

        w = np.linalg.norm(p[:, cfg.SHOULDER_L] - p[:, cfg.SHOULDER_R], axis=-1)
        med_w = float(np.median(w))
        shoulder_widths.append(med_w)
        if med_w < cfg.MIN_SHOULDER_SCALE * 10:
            tiny_scale += 1

        # frame-to-frame jump, in shoulder-widths (teleporting joints)
        if len(p) > 1 and med_w > 1e-6:
            d = np.linalg.norm(np.diff(p, axis=0), axis=-1).mean(axis=-1) / med_w
            jumps.append(float(np.percentile(d, 99)))

    n = len(idxs)
    print("\n" + "=" * 56)
    print(f" DATA QUALITY  ({n} sampled sequences)")
    print("=" * 56)
    print(f"sequences containing NaN      : {n_nan}   {'<-- MUST FIX' if n_nan else 'OK'}")
    print(f"sequences containing Inf      : {n_inf}   {'<-- MUST FIX' if n_inf else 'OK'}")
    print(f"entirely-zero sequences       : {n_allzero_seq}   "
          f"{'<-- MUST FIX' if n_allzero_seq else 'OK'}")
    print(f"near-zero shoulder width      : {tiny_scale}   "
          f"{'<-- normalization will be unstable' if tiny_scale else 'OK'}")
    if zero_frame_frac:
        zf = np.array(zero_frame_frac)
        print(f"\nblank frames per sequence     : mean {100*zf.mean():.2f}%  "
              f"max {100*zf.max():.2f}%")
        bad = int((zf > 0.2).sum())
        print(f"  sequences >20% blank frames : {bad}   "
              f"{'<-- consider filtering these out' if bad else 'OK'}")
    if shoulder_widths:
        sw = np.array(shoulder_widths)
        print(f"\nshoulder width (raw units)    : min {sw.min():.4f}  "
              f"median {np.median(sw):.4f}  max {sw.max():.4f}")
        print(f"  ratio max/min               : {sw.max()/max(sw.min(),1e-9):.1f}x  "
              f"(this is the signer size variation normalization removes)")
    if jumps:
        j = np.array(jumps)
        print(f"\nframe-to-frame jump (p99, shoulder-widths): "
              f"median {np.median(j):.3f}  max {j.max():.3f}")
        print(f"  sequences with jumps >1.0 shoulder-width : {int((j>1.0).sum())}   "
              f"{'<-- tracking glitches' if (j>1.0).sum() else 'OK'}")
    print("=" * 56)

    if n_nan or n_inf or n_allzero_seq:
        print("STOP: fix the flagged files before training. NaN/Inf will make every")
        print("gradient NaN within one step, and the run will silently produce nothing.")
    else:
        print("No blocking issues found — safe to train.")


quality_check(train_ds)


# **5. Text Encoder (BanglaBERT + Signer Conditioning)**

Loads pretrained BanglaBERT and projects to `d_model`. When
`USE_SIGNER_EMBEDDING` is on, a learned per-signer vector is added to every
token position, letting the model condition on *who* is signing instead of
averaging five people's styles into one blurry output.

In [ ]:
import torch.nn as nn
from transformers import AutoModel


class GlossTextEncoder(nn.Module):
    def __init__(self, d_model=cfg.D_MODEL, freeze_bert=cfg.FREEZE_BANGLABERT,
                 use_signer=cfg.USE_SIGNER_EMBEDDING, num_signers=cfg.NUM_SIGNERS):
        super().__init__()
        self.bert = AutoModel.from_pretrained(cfg.BANGLABERT_MODEL)
        bert_hidden = self.bert.config.hidden_size
        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.proj = nn.Linear(bert_hidden, d_model)
        self.use_signer = use_signer
        if use_signer:
            self.signer_emb = nn.Embedding(num_signers, d_model)

        self.refine = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead=8, dim_feedforward=1024,
                                       batch_first=True),
            num_layers=2,
        )

    def forward(self, input_ids, attention_mask, signer_ids=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        x = self.proj(out)
        if self.use_signer:
            if signer_ids is None:
                raise ValueError("USE_SIGNER_EMBEDDING is on but signer_ids was not provided.")
            x = x + self.signer_emb(signer_ids).unsqueeze(1)
        x = self.refine(x, src_key_padding_mask=(attention_mask == 0))
        return x

    def bert_parameters(self):
        return list(self.bert.parameters())

    def non_bert_parameters(self):
        bert_ids = {id(p) for p in self.bert.parameters()}
        return [p for p in self.parameters() if id(p) not in bert_ids]


# **6. Temporal Positional Encoding**

In [ ]:
import math


class TemporalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=3000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        T = x.size(1)
        if T > self.pe.size(1):
            raise ValueError(f"Sequence length {T} exceeds positional table {self.pe.size(1)}.")
        return x + self.pe[:, :T, :]


# **7. Duration Predictor & Length Regulator**

The predictor regresses `log(1 + frames)` per token,
training teacher-forces ground-truth durations so predicted and target sequences
share a time axis, and inference uses the model's own predictions. Frames are
allocated uniformly across real tokens as a stand-in for a forced aligner (BdSL
has no aligner and no frame-level gloss timing).

Limitation: uniform allocation assumes every
gloss token takes equal time, which is linguistically false. It is the honest
best available option given the annotations, and a per-token alignment model is
natural future work.

In [ ]:
class DurationPredictor(nn.Module):
    def __init__(self, d_model=cfg.D_MODEL, filter_size=256, kernel_size=3, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(d_model, filter_size, kernel_size, padding=kernel_size // 2)
        self.norm1 = nn.LayerNorm(filter_size)
        self.conv2 = nn.Conv1d(filter_size, filter_size, kernel_size, padding=kernel_size // 2)
        self.norm2 = nn.LayerNorm(filter_size)
        self.linear_layer = nn.Linear(filter_size, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, encoder_output, mask=None):
        x = encoder_output.transpose(1, 2)
        x = self.relu(self.conv1(x))
        x = self.norm1(x.transpose(1, 2)).transpose(1, 2)
        x = self.dropout(x)
        x = self.relu(self.conv2(x))
        x = self.norm2(x.transpose(1, 2))
        x = self.dropout(x)
        log_dur = self.linear_layer(x).squeeze(-1)
        if mask is not None:
            log_dur = log_dur.masked_fill(mask == 0, 0.0)
        return log_dur


def compute_uniform_target_durations(attention_mask, target_lengths):
    """Integer frames per token, summing exactly to each sample's true length."""
    B, L = attention_mask.shape
    durations = torch.zeros(B, L, dtype=torch.long, device=attention_mask.device)
    for i in range(B):
        n_tok = int(attention_mask[i].sum().item())
        if n_tok == 0:
            continue
        total = int(target_lengths[i].item())
        base, rem = total // n_tok, total % n_tok
        tok_d = torch.full((n_tok,), base, dtype=torch.long, device=attention_mask.device)
        if rem > 0:
            tok_d[:rem] += 1
        durations[i, attention_mask[i].nonzero(as_tuple=True)[0]] = tok_d
    return durations


class LengthRegulator(nn.Module):
    def forward(self, encoder_output, durations):
        B = encoder_output.size(0)
        expanded = []
        for i in range(B):
            d = durations[i].clamp(min=0)
            if d.sum() == 0:
                expanded.append(torch.zeros((1, encoder_output.size(-1)),
                                            device=encoder_output.device,
                                            dtype=encoder_output.dtype))
            else:
                expanded.append(torch.repeat_interleave(encoder_output[i], d, dim=0))
        return pad_sequence(expanded, batch_first=True, padding_value=0.0)


def make_frame_mask(lengths, max_T, device):
    """(B, T) float mask, 1.0 on real frames."""
    ar = torch.arange(max_T, device=device).unsqueeze(0)
    return (ar < lengths.unsqueeze(1).to(device)).float()


# **8. Diffusion Pose Generator**

x0-parameterized diffusion with `DIFFUSION_TIMESTEPS = 250`. Fewer timesteps means each one carries more of the
training signal, which matters at this dataset size — 1000 steps spreads a
limited number of gradient updates very thinly.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=time.device) * -emb)
        emb = time[:, None].float() * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)


class PoseDenoiser(nn.Module):
    def __init__(self, pose_dim=cfg.POSE_DIM, cond_dim=cfg.D_MODEL,
                 time_dim=cfg.D_MODEL, num_layers=4, nhead=8):
        super().__init__()
        self.pose_proj = nn.Linear(pose_dim, cond_dim)
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim), nn.GELU(),
            nn.Linear(time_dim, cond_dim),
        )
        self.transformer_decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=cond_dim, nhead=nhead, batch_first=True),
            num_layers=num_layers,
        )
        self.output_proj = nn.Linear(cond_dim, pose_dim)

    def forward(self, x_t, time, cond):
        B, T, _ = x_t.shape
        x = self.pose_proj(x_t) + self.time_mlp(time).unsqueeze(1).expand(-1, T, -1)
        x = self.transformer_decoder(tgt=x, memory=cond)
        return self.output_proj(x).view(B, T, cfg.NUM_JOINTS, cfg.COORD_DIM)


class DiffusionG2P(nn.Module):
    def __init__(self, timesteps=cfg.DIFFUSION_TIMESTEPS):
        super().__init__()
        self.denoiser = PoseDenoiser()
        self.timesteps = timesteps
        betas = torch.linspace(1e-4, 0.02, timesteps)
        alphas = 1.0 - betas
        self.register_buffer('betas', betas)
        self.register_buffer('alphas', alphas)
        self.register_buffer('alpha_bars', torch.cumprod(alphas, dim=0))
        # Learned stand-in for "no text condition", used by classifier-free guidance.
        self.null_cond = nn.Parameter(torch.randn(1, 1, cfg.D_MODEL) * 0.02)

    def _maybe_drop_cond(self, cond, p):
        if p <= 0:
            return cond
        B = cond.size(0)
        drop = (torch.rand(B, device=cond.device) < p).view(B, 1, 1)
        return torch.where(drop, self.null_cond.expand_as(cond), cond)

    def forward(self, x0, cond, cfg_dropout=cfg.CFG_DROPOUT):
        B, T, J, C = x0.shape
        t = torch.randint(0, self.timesteps, (B,), device=x0.device).long()
        ab = self.alpha_bars[t].view(B, 1, 1, 1)
        noise = torch.randn_like(x0)
        x_t = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise
        cond = self._maybe_drop_cond(cond, cfg_dropout)
        return self.denoiser(x_t.view(B, T, J * C), t, cond)

    def _smooth_step(self, x_t, t_b, cond, lr, iters, frame_mask):
        """Test-time latent guidance (idea from MotionFlow, Meral et al., AAAI 2026):
        take a few gradient steps on the noisy sample so the x0 it implies has
        less frame-to-frame acceleration, then let normal denoising continue.
        Training-free; touches no weights. The step is normalised per sample so
        `lr` is the mean absolute change per coordinate."""
        B, T = x_t.shape[:2]
        if T < 3:
            return x_t
        w = None
        if frame_mask is not None:
            w = (frame_mask[:, 2:] * frame_mask[:, 1:-1] * frame_mask[:, :-2]).view(B, T - 2, 1)
        with torch.enable_grad():
            for _ in range(iters):
                xg = x_t.detach().clone().requires_grad_(True)
                p0 = self.denoiser(xg.view(B, T, cfg.POSE_DIM), t_b, cond).view(B, T, cfg.POSE_DIM)
                sq = (p0[:, 2:] - 2 * p0[:, 1:-1] + p0[:, :-2]) ** 2
                if w is not None:
                    loss = (sq * w).sum() / (w.sum() * cfg.POSE_DIM).clamp(min=1.0)
                else:
                    loss = sq.mean()
                g, = torch.autograd.grad(loss, xg)
                scale = g.abs().mean(dim=(1, 2, 3), keepdim=True) + 1e-12
                x_t = (xg - lr * g / scale).detach()
        return x_t

    @torch.no_grad()
    def sample(self, cond, num_steps=cfg.SAMPLING_STEPS, guidance=cfg.CFG_SCALE,
               smooth_lr=0.0, smooth_steps=20, smooth_iters=3, frame_mask=None):
        B, T, _ = cond.shape
        dev = cond.device
        x_t = torch.randn(B, T, cfg.NUM_JOINTS, cfg.COORD_DIM, device=dev)
        steps = torch.linspace(self.timesteps - 1, 0, num_steps, device=dev).long()
        null = self.null_cond.expand_as(cond)

        for i, t_val in enumerate(steps):
            t_b = torch.full((B,), int(t_val.item()), device=dev, dtype=torch.long)
            if smooth_lr > 0 and i < smooth_steps:
                x_t = self._smooth_step(x_t, t_b, cond, smooth_lr, smooth_iters, frame_mask)
            ab_t = self.alpha_bars[t_val]
            xf = x_t.view(B, T, cfg.POSE_DIM)
            pred_x0 = self.denoiser(xf, t_b, cond)
            if guidance != 1.0:
                # push away from what the model would produce with no gloss at all
                pred_uncond = self.denoiser(xf, t_b, null)
                pred_x0 = pred_uncond + guidance * (pred_x0 - pred_uncond)
            if i == len(steps) - 1:
                x_t = pred_x0
                break
            ab_next = self.alpha_bars[steps[i + 1]]
            eps = (x_t - torch.sqrt(ab_t) * pred_x0) / torch.sqrt(1 - ab_t + 1e-8)
            x_t = torch.sqrt(ab_next) * pred_x0 + torch.sqrt(1 - ab_next) * eps
        return x_t


# **9. Pose Refiner**

In [ ]:
class PoseRefiner(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.temporal_smooth = nn.AvgPool1d(kernel_size=kernel_size, stride=1,
                                            padding=kernel_size // 2)
        self.anchors = [cfg.SHOULDER_L, cfg.SHOULDER_R, cfg.ELBOW_L,
                        cfg.ELBOW_R, cfg.WRIST_L, cfg.WRIST_R]
        self.alpha = 0.80

    def forward(self, raw_poses):
        B, T, J, C = raw_poses.shape
        x = raw_poses.view(B, T, J * C).transpose(1, 2)
        smoothed = self.temporal_smooth(x).transpose(1, 2).view(B, T, J, C)
        out = smoothed.clone()
        for idx in self.anchors:
            out[:, :, idx, :] = smoothed[:, :, idx, :] * self.alpha + \
                                raw_poses[:, :, idx, :] * (1 - self.alpha)
        return out


# **9b. Comparison Architectures (Progressive Transformer & NAT + MSE)**

Two further generators, chosen to isolate two specific questions.

**Progressive Transformer** (Saunders et al., *Progressive Transformers for End-to-End Sign
Language Production*, ECCV 2020) asks whether the non-autoregressive architecture itself matters.
It is autoregressive: a causal Transformer decoder predicts one pose frame at a time, plus a scalar
**counter** in [0, 1] tracking progress through the sentence, and generation stops once the
counter saturates. There is no duration predictor; the model finds its own stopping point.
Training is a single parallel forward pass (teacher forcing under a causal mask); only inference
is sequential.

**NAT + MSE** asks whether a gain comes from the diffusion generator or from the Soft-DTW +
velocity + bone loss stack. It reuses the same text encoder, duration predictor and length
regulator as the diffusion model and replaces only the generator with a single feed-forward
regression head over the length-regulated features, trained with plain per-frame MSE (no
Soft-DTW, velocity or bone term). It is exactly as non-autoregressive and as parallel at inference
as the diffusion model; only the generation head and the loss differ.


In [ ]:
class ProgressivePoseDecoder(nn.Module):
    """Autoregressive sign pose decoder (Saunders et al., ECCV 2020 style).

    Each step attends causally to its own previously generated frames and
    cross-attends to the encoded gloss, predicting the next pose frame plus a
    "counter" in [0, 1]. Generation stops once the counter saturates, so
    sequence length is discovered by the model itself rather than by a
    separate duration predictor.
    """
    def __init__(self, pose_dim=cfg.POSE_DIM, cond_dim=cfg.D_MODEL, num_layers=4, nhead=8):
        super().__init__()
        self.pose_dim = pose_dim
        self.frame_proj = nn.Linear(pose_dim + 1, cond_dim)   # pose + counter-so-far
        self.start_token = nn.Parameter(torch.zeros(1, 1, pose_dim + 1))
        self.pos_enc = TemporalPositionalEncoding(cond_dim)
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(cond_dim, nhead=nhead, dim_feedforward=1024,
                                       batch_first=True),
            num_layers=num_layers,
        )
        self.out_proj = nn.Linear(cond_dim, pose_dim + 1)     # next pose + next counter

    @staticmethod
    def _causal_mask(T, device):
        return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

    def forward(self, target_poses, counters, cond, cond_mask=None):
        """Teacher-forced training pass: predict frame t from frames < t.

        target_poses: (B,T,J,C); counters: (B,T) in [0,1]; cond: (B,L,d_model).
        Returns pred_pose (B,T,J,C), pred_counter (B,T).
        """
        B, T, J, C = target_poses.shape
        flat = target_poses.reshape(B, T, J * C)
        frame_ctr = torch.cat([flat, counters.unsqueeze(-1)], dim=-1)
        shifted = torch.cat([self.start_token.expand(B, -1, -1).to(flat.dtype), frame_ctr[:, :-1]], dim=1)
        x = self.pos_enc(self.frame_proj(shifted))
        mask = self._causal_mask(T, x.device)
        key_padding = (cond_mask == 0) if cond_mask is not None else None
        h = self.decoder(tgt=x, memory=cond, tgt_mask=mask,
                         memory_key_padding_mask=key_padding)
        out = self.out_proj(h)
        pred_pose = out[..., :-1].view(B, T, J, C)
        pred_ctr = out[..., -1]
        return pred_pose, pred_ctr

    @torch.no_grad()
    def generate(self, cond, cond_mask=None, max_frames=cfg.MAX_GEN_FRAMES,
                counter_threshold=0.999):
        """Autoregressive sampling, one frame at a time — the sequential,
        slower-at-inference path the architecture comparison table measures."""
        B = cond.size(0)
        dev = cond.device
        cur = self.start_token.expand(B, -1, -1).clone().to(cond.dtype)
        frames, ctrs = [], []
        key_padding = (cond_mask == 0) if cond_mask is not None else None
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        for t in range(max_frames):
            x = self.pos_enc(self.frame_proj(cur))
            mask = self._causal_mask(cur.size(1), dev)
            h = self.decoder(tgt=x, memory=cond, tgt_mask=mask,
                             memory_key_padding_mask=key_padding)
            nxt = self.out_proj(h[:, -1:, :])
            frames.append(nxt[..., :-1])
            ctr = nxt[..., -1].squeeze(-1).clamp(0, 1)
            ctrs.append(ctr)
            done = done | (ctr >= counter_threshold)
            cur = torch.cat([cur, nxt], dim=1)
            if bool(done.all()) and t >= 1:
                break
        pose_seq = torch.cat(frames, dim=1).view(B, -1, cfg.NUM_JOINTS, cfg.COORD_DIM)
        ctr_seq = torch.stack(ctrs, dim=1)
        T_out = pose_seq.size(1)
        lengths = torch.full((B,), T_out, dtype=torch.long, device=dev)
        for b in range(B):
            hit = (ctr_seq[b] >= counter_threshold).nonzero(as_tuple=True)[0]
            if len(hit) > 0:
                lengths[b] = int(hit[0].item()) + 1
        return pose_seq, ctr_seq, lengths


def compute_counter_targets(lengths, max_T, device):
    """Per-frame progress target in (0,1], linear from 1/len to 1 at the last
    valid frame. Padded positions are ignored by the frame mask in the loss."""
    ar = torch.arange(1, max_T + 1, device=device).float().unsqueeze(0)
    denom = lengths.view(-1, 1).float().clamp(min=1).to(device)
    return (ar / denom).clamp(max=1.0)


class NATRegressionHead(nn.Module):
    """Non-autoregressive, non-diffusion pose head: a Transformer-encoder
    stack maps the length-regulated encoder features directly to pose
    coordinates in one parallel pass, trained with plain per-frame MSE. Same
    input/output shape and same parallel inference cost as the diffusion
    generator — only the generation mechanism and training objective differ.
    """
    def __init__(self, cond_dim=cfg.D_MODEL, pose_dim=cfg.POSE_DIM, num_layers=4, nhead=8):
        super().__init__()
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(cond_dim, nhead=nhead, dim_feedforward=1024,
                                       batch_first=True),
            num_layers=num_layers,
        )
        self.out_proj = nn.Linear(cond_dim, pose_dim)

    def forward(self, expanded_cond, frame_mask=None):
        key_padding = (frame_mask == 0) if frame_mask is not None else None
        h = self.encoder(expanded_cond, src_key_padding_mask=key_padding)
        B, T, _ = h.shape
        return self.out_proj(h).view(B, T, cfg.NUM_JOINTS, cfg.COORD_DIM)


# **10. Loss Function (Masked & Length-Normalized)**

**Padding is masked out.** Predictions are zeroed on padded frames and per-frame terms are averaged
over valid frames only, so regions with no ground truth never count toward the objective.

**Soft-DTW is divided by sequence length.** Soft-DTW accumulates cost along an alignment path, so
its raw magnitude grows with the number of frames; dividing by length keeps the loss comparable
across batches of different video length.


In [ ]:
import torch.nn.functional as F
from tslearn.metrics import SoftDTWLossPyTorch


class G2PLossStack(nn.Module):
    def __init__(self, gamma=cfg.SOFT_DTW_GAMMA, bone_weight=cfg.BONE_WEIGHT,
                 hand_weight=cfg.HAND_WEIGHT, duration_weight=cfg.DURATION_WEIGHT,
                 leg_weight=cfg.LEG_WEIGHT, velocity_weight=cfg.VELOCITY_WEIGHT):
        super().__init__()
        self.soft_dtw = SoftDTWLossPyTorch(gamma=gamma)
        self.bone_weight = bone_weight
        self.hand_weight = hand_weight
        self.duration_weight = duration_weight
        self.leg_weight = leg_weight
        self.velocity_weight = velocity_weight
        self.bone_pairs = cfg.BONE_PAIRS

    def joint_weights(self, device):
        w = torch.ones(cfg.NUM_JOINTS, device=device)
        w[cfg.LEG_START:cfg.LEG_END] = self.leg_weight   # extrapolated -> quiet
        w[cfg.HAND_START:cfg.HAND_END] = self.hand_weight
        return w

    def _bone_loss(self, pred, target, frame_mask):
        m = frame_mask                                   # (B, T)
        denom = m.sum().clamp(min=1.0)
        total = 0.0
        for j1, j2 in self.bone_pairs:
            p_len = torch.norm(pred[:, :, j1, :] - pred[:, :, j2, :], dim=-1)
            t_len = torch.norm(target[:, :, j1, :] - target[:, :, j2, :], dim=-1)
            # per-sample mean target bone length over VALID frames only
            t_ref = (t_len * m).sum(dim=1, keepdim=True) / m.sum(dim=1, keepdim=True).clamp(min=1.0)
            total = total + (((p_len - t_ref.detach()) ** 2) * m).sum() / denom
        return total / len(self.bone_pairs)

    def _velocity_loss(self, pred, target, frame_mask, jw):
        """Match frame-to-frame motion, not just pose. A frozen prediction has
        zero velocity everywhere, so this term charges it in full."""
        pv = (pred[:, 1:] - pred[:, :-1]) * jw
        tv = (target[:, 1:] - target[:, :-1]) * jw
        m = (frame_mask[:, 1:] * frame_mask[:, :-1]).unsqueeze(-1).unsqueeze(-1)
        denom = (m.sum() * cfg.NUM_JOINTS * cfg.COORD_DIM).clamp(min=1.0)
        return (((pv - tv) ** 2) * m).sum() / denom

    def _duration_loss(self, pred_log_dur, target_dur, attention_mask):
        tgt = torch.log(target_dur.float() + 1.0)
        m = attention_mask.float()
        return (((pred_log_dur - tgt) ** 2) * m).sum() / m.sum().clamp(min=1.0)

    def forward(self, pred_poses, target_poses, frame_mask, lengths,
                pred_log_dur=None, target_dur=None, attention_mask=None):
        fm4 = frame_mask.unsqueeze(-1).unsqueeze(-1)     # (B, T, 1, 1)
        pred_poses = pred_poses * fm4                    # kill padded predictions
        target_poses = target_poses * fm4

        jw = self.joint_weights(pred_poses.device).view(1, 1, cfg.NUM_JOINTS, 1)

        B, T, J, C = pred_poses.shape
        p_flat = (pred_poses * jw).view(B, T, J * C).float()
        t_flat = (target_poses * jw).view(B, T, J * C).float()

        dtw_per_item = self.soft_dtw(p_flat, t_flat)                  # (B,)
        dtw_loss = (dtw_per_item / lengths.to(dtw_per_item.device).float().clamp(min=1.0)).mean()

        bone_loss = self._bone_loss(pred_poses.float(), target_poses.float(), frame_mask)
        vel_loss = self._velocity_loss(pred_poses.float(), target_poses.float(), frame_mask, jw)
        total = dtw_loss + self.bone_weight * bone_loss + self.velocity_weight * vel_loss

        dur_loss = torch.zeros((), device=pred_poses.device)
        if pred_log_dur is not None and target_dur is not None and attention_mask is not None:
            dur_loss = self._duration_loss(pred_log_dur.float(), target_dur, attention_mask)
            total = total + self.duration_weight * dur_loss

        return total, dtw_loss, bone_loss, dur_loss, vel_loss


# **10b. Loss Functions of the Comparison Models**

Deliberately the simplest reasonable objective for each of these models — neither
gets the velocity term, the bone term, or DTW alignment. That asymmetry is the
point: it is what the comparison in Section 16c is measuring.

In [ ]:
class MSEPoseLoss(nn.Module):
    """Plain masked per-frame MSE for the NAT + MSE model. No DTW alignment
    (so no robustness to duration-predictor error), no velocity term (so
    nothing specifically penalizes a frozen/mean-pose collapse), no bone term."""
    def __init__(self, duration_weight=cfg.DURATION_WEIGHT, variance_weight=0.0):
        super().__init__()
        self.duration_weight = duration_weight
        # Temporal-variance matching: plain MSE regresses toward the mean pose, so
        # its output moves less than the target (motion ratio ~0.8). This penalises
        # the squared gap between predicted and true per-joint standard deviation
        # over time, computed per sequence on valid frames only. Legs are down-
        # weighted as elsewhere (extrapolated landmarks). 0 disables it.
        self.variance_weight = variance_weight

    def _variance_loss(self, pred, target, frame_mask):
        m = frame_mask.unsqueeze(-1).unsqueeze(-1)                    # (B, T, 1, 1)
        n = m.sum(dim=1, keepdim=True).clamp(min=2.0)                 # (B, 1, 1, 1)

        def tstd(x):
            mu = (x * m).sum(dim=1, keepdim=True) / n
            var = (((x - mu) ** 2) * m).sum(dim=1) / n.squeeze(1)     # (B, J, C)
            return (var + 1e-8).sqrt()

        w = torch.ones(cfg.NUM_JOINTS, device=pred.device)
        w[cfg.LEG_START:cfg.LEG_END] = cfg.LEG_WEIGHT
        return (((tstd(pred.float()) - tstd(target.float())) ** 2) * w.view(1, -1, 1)).mean()

    def _duration_loss(self, pred_log_dur, target_dur, attention_mask):
        tgt = torch.log(target_dur.float() + 1.0)
        m = attention_mask.float()
        return (((pred_log_dur - tgt) ** 2) * m).sum() / m.sum().clamp(min=1.0)

    def forward(self, pred_poses, target_poses, frame_mask, pred_log_dur=None,
               target_dur=None, attention_mask=None):
        fm = frame_mask.unsqueeze(-1).unsqueeze(-1)
        diff = (pred_poses - target_poses) * fm
        denom = (frame_mask.sum() * cfg.NUM_JOINTS * cfg.COORD_DIM).clamp(min=1.0)
        mse = (diff ** 2).sum() / denom
        total = mse
        if self.variance_weight > 0:
            total = total + self.variance_weight * self._variance_loss(
                pred_poses, target_poses, frame_mask)
        dur_loss = torch.zeros((), device=pred_poses.device)
        if pred_log_dur is not None and target_dur is not None and attention_mask is not None:
            dur_loss = self._duration_loss(pred_log_dur.float(), target_dur, attention_mask)
            total = total + self.duration_weight * dur_loss
        return total, mse, dur_loss


class ProgressiveLoss(nn.Module):
    """Masked MSE on pose + counter, per Saunders et al. (2020)."""
    def __init__(self, counter_weight=1.0):
        super().__init__()
        self.counter_weight = counter_weight

    def forward(self, pred_pose, pred_ctr, target_pose, target_ctr, frame_mask):
        fm4 = frame_mask.unsqueeze(-1).unsqueeze(-1)
        denom = (frame_mask.sum() * cfg.NUM_JOINTS * cfg.COORD_DIM).clamp(min=1.0)
        pose_mse = (((pred_pose - target_pose) * fm4) ** 2).sum() / denom
        ctr_denom = frame_mask.sum().clamp(min=1.0)
        ctr_mse = (((pred_ctr - target_ctr) * frame_mask) ** 2).sum() / ctr_denom
        total = pose_mse + self.counter_weight * ctr_mse
        return total, pose_mse, ctr_mse


# **11. Metrics: DTW-MJE, Multi-Threshold PCK, Length Ratio**

`DTW-MJE` is the metric the SLRTP2025 challenge standardized on, and it replaces naive frame-index truncation. The reason matters: if the duration predictor is even
slightly off, frame *i* of the prediction corresponds to a different moment of
the sign than frame *i* of the target, so index-wise comparison charges the pose
model for the timing model's error. DTW finds the optimal monotonic alignment
first, then measures pose error along it.

`dtw_path` from tslearn is numba-compiled, so the O(T1xT2) dynamic program is
fast enough to run over the whole test set.

`length_ratio` (predicted frames / true frames) isolates *timing* quality on its
own — a ratio far from 1.0 means the duration predictor is the problem, not the
pose generator.

In [ ]:
from tslearn.metrics import dtw_path

PCK_THRESHOLDS = (0.05, 0.1, 0.2, 0.5)


def pose_metrics(pred, target, thresholds=PCK_THRESHOLDS):
    """DTW-MJE + multi-threshold PCK from a SINGLE alignment.

    pred: (T1, J, 3) np.ndarray, target: (T2, J, 3) np.ndarray.
    The DTW path is the expensive part (O(T1*T2)), so it is computed once and
    reused for every metric rather than recomputed per metric.
    """
    T1, T2 = pred.shape[0], target.shape[0]
    path, _ = dtw_path(pred.reshape(T1, -1).astype(np.float64),
                       target.reshape(T2, -1).astype(np.float64))

    mje_terms = []
    hits = {f"PCK@{t}": 0.0 for t in thresholds}
    n_joint_obs = 0

    for i, j in path:
        err = np.linalg.norm(pred[i] - target[j], axis=-1)      # (J,)
        mje_terms.append(err.mean())

        scale = np.linalg.norm(target[j, cfg.SHOULDER_L] - target[j, cfg.SHOULDER_R])
        norm_err = err / max(scale, 1e-6)
        for t in thresholds:
            hits[f"PCK@{t}"] += float((norm_err < t).sum())
        n_joint_obs += err.shape[0]

    out = {'dtw_mje': float(np.mean(mje_terms))}
    for t in thresholds:
        k = f"PCK@{t}"
        out[k] = 100.0 * hits[k] / max(n_joint_obs, 1)

    # Total travelled distance exposes a frozen output. A static prediction
    # scores ~0 here regardless of how respectable its DTW-MJE looks.
    def travelled(p):
        if len(p) < 2:
            return 0.0
        return float(np.linalg.norm(np.diff(p, axis=0), axis=-1).mean(axis=-1).sum())

    tp, tt = travelled(pred), travelled(target)
    out['motion_ratio'] = tp / max(tt, 1e-8)      # 1.0 = matches ground-truth motion

    # Mean second difference (frame-to-frame acceleration). Motion ratio alone
    # cannot tell large smooth movements from high-frequency jitter; a jitter
    # ratio well above the motion ratio means the extra motion is noise.
    def accel(p):
        if len(p) < 3:
            return 0.0
        return float(np.linalg.norm(np.diff(p, n=2, axis=0), axis=-1).mean())

    out['jitter_ratio'] = accel(pred) / max(accel(target), 1e-8)
    return out


def summarize_metrics(records):
    """records: list of dicts from evaluate_generation."""
    if not records:
        return {}
    keys = records[0].keys()
    return {k: float(np.mean([r[k] for r in records])) for k in keys}


def print_metrics(name, m, n_samples):
    print("\n" + "=" * 56)
    print(f" {name}   (n = {n_samples} videos)")
    print("=" * 56)
    print(f"DTW-MJE (primary)     : {m.get('dtw_mje', float('nan')):.4f}")
    for t in PCK_THRESHOLDS:
        k = f"PCK@{t}"
        if k in m:
            print(f"{k:<22}: {m[k]:.2f}%")
    print(f"Length ratio (pred/gt): {m.get('length_ratio', float('nan')):.3f}   (1.000 is perfect)")
    mr = m.get('motion_ratio', float('nan'))
    flag = ""
    if mr == mr:
        flag = ("  <-- FROZEN OUTPUT / mode collapse" if mr < 0.5
                else ("  <-- over-moving / jittery" if mr > 1.6 else "  (healthy)"))
    print(f"Motion ratio (pred/gt): {mr:.3f}{flag}")
    print(f"Jitter ratio (accel pred/gt): {m.get('jitter_ratio', float('nan')):.3f}   "
          "(>> motion ratio means the extra motion is high-frequency noise)")
    print("=" * 56)
    print("Units: shoulder-widths (poses normalized).")


# **12. Initialization & Smoke Test**

Three architectures, each with its own text encoder instance (so none of them
share trained weights — they must be trained fully independently to be a fair
comparison). `length_regulator`, `temporal_pos_encoder` and `pose_refiner`
have no learnable parameters, so those three are genuinely shared utilities,
not shared trained state.

In [ ]:
set_seed(cfg.SEED)

# ---- Diffusion model: NAT + Diffusion + Velocity ----
text_encoder = GlossTextEncoder().to(device)
duration_predictor = DurationPredictor().to(device)
length_regulator = LengthRegulator().to(device)
temporal_pos_encoder = TemporalPositionalEncoding(cfg.D_MODEL).to(device)
diffusion_g2p = DiffusionG2P().to(device)
pose_refiner = PoseRefiner().to(device)
loss_fn = G2PLossStack().to(device)

# ---- Progressive Transformer (autoregressive) ----
text_encoder_prog = GlossTextEncoder().to(device)
progressive_decoder = ProgressivePoseDecoder().to(device)
progressive_loss_fn = ProgressiveLoss().to(device)

# ---- NAT + MSE (non-autoregressive, no diffusion) ----
text_encoder_nat_mse = GlossTextEncoder().to(device)
duration_predictor_nat_mse = DurationPredictor().to(device)
nat_mse_head = NATRegressionHead().to(device)
mse_loss_fn = MSEPoseLoss(variance_weight=cfg.VARIANCE_WEIGHT).to(device)


def count_trainable(*modules):
    return sum(p.numel() for m in modules for p in m.parameters() if p.requires_grad)


print("Trainable parameters:")
print(f"  NAT + Diffusion + Velocity            : "
     f"{count_trainable(text_encoder, duration_predictor, diffusion_g2p)/1e6:.1f}M")
print(f"  Progressive Transformer               : "
     f"{count_trainable(text_encoder_prog, progressive_decoder)/1e6:.1f}M")
print(f"  NAT + MSE                             : "
     f"{count_trainable(text_encoder_nat_mse, duration_predictor_nat_mse, nat_mse_head)/1e6:.1f}M")

print("\nRunning smoke test (diffusion model)...")

for batch in train_loader:
    input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
        b.to(device) for b in batch
    ]
    enc = text_encoder(input_ids, attention_mask, signer_ids)
    log_dur = duration_predictor(enc, attention_mask)
    tgt_dur = compute_uniform_target_durations(attention_mask, lengths)
    expanded = temporal_pos_encoder(length_regulator(enc, tgt_dur))
    pred_x0 = diffusion_g2p(target_poses, expanded)
    fm = make_frame_mask(lengths, target_poses.size(1), device)

    total, dtw_l, bone_l, dur_l, vel_l = loss_fn(
        pred_x0, target_poses, fm, lengths, log_dur, tgt_dur, attention_mask
    )

    print("\n--- Smoke Test OK ---")
    print(f"target_poses     : {tuple(target_poses.shape)}")
    print(f"expanded cond    : {tuple(expanded.shape)}  (T must match target)")
    print(f"pred_x0          : {tuple(pred_x0.shape)}")
    print(f"true lengths     : {lengths.tolist()}")
    print(f"loss total {total.item():.4f} | dtw {dtw_l.item():.4f} | "
          f"bone {bone_l.item():.4f} | dur {dur_l.item():.4f} | vel {vel_l.item():.4f}")

    # How the objective is actually distributed across the body, after weighting.
    with torch.no_grad():
        w = loss_fn.joint_weights(device)
        var = target_poses.float().var(dim=(0, 1))            # (J, 3)
        energy = (var.sum(dim=-1) * w ** 2)                   # (J,)
        tot_e = energy.sum().clamp(min=1e-9)
        spans = [("body ex-legs", 0, cfg.LEG_START), ("legs (extrapolated)", cfg.LEG_START, cfg.LEG_END),
                 ("hands", cfg.HAND_START, cfg.HAND_END), ("extra", cfg.HAND_END, cfg.NUM_JOINTS)]
        print("\nShare of the Soft-DTW objective by region:")
        for name, a, b in spans:
            print(f"  {name:<22}{100*energy[a:b].sum().item()/tot_e.item():6.1f}%   "
                  f"(joint weight {w[a].item():.2f})")
        print(f"  LEG_WEIGHT={cfg.LEG_WEIGHT} -> set 1.0 to see the unweighted split.")
    assert expanded.size(1) == target_poses.size(1), "Teacher forcing length mismatch!"
    break

print("\nRunning smoke test (Progressive Transformer)...")
for batch in train_loader:
    input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
        b.to(device) for b in batch
    ]
    enc = text_encoder_prog(input_ids, attention_mask, signer_ids)
    ctr_tgt = compute_counter_targets(lengths, target_poses.size(1), device)
    pred_pose, pred_ctr = progressive_decoder(target_poses, ctr_tgt, enc, cond_mask=attention_mask)
    fm = make_frame_mask(lengths, target_poses.size(1), device)
    total, pose_mse, ctr_mse = progressive_loss_fn(pred_pose, pred_ctr, target_poses, ctr_tgt, fm)
    print(f"pred_pose {tuple(pred_pose.shape)} | pred_ctr {tuple(pred_ctr.shape)} | "
         f"loss {total.item():.4f} (pose_mse {pose_mse.item():.4f}, ctr_mse {ctr_mse.item():.4f})")
    assert pred_pose.shape == target_poses.shape
    break

print("\nRunning smoke test (NAT + MSE)...")
for batch in train_loader:
    input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
        b.to(device) for b in batch
    ]
    enc = text_encoder_nat_mse(input_ids, attention_mask, signer_ids)
    log_dur = duration_predictor_nat_mse(enc, attention_mask)
    tgt_dur = compute_uniform_target_durations(attention_mask, lengths)
    expanded = temporal_pos_encoder(length_regulator(enc, tgt_dur))
    fm = make_frame_mask(lengths, target_poses.size(1), device)
    pred = nat_mse_head(expanded, fm)
    total, mse, dur_l = mse_loss_fn(pred, target_poses, fm, log_dur, tgt_dur, attention_mask)
    print(f"pred {tuple(pred.shape)} | loss {total.item():.4f} "
         f"(mse {mse.item():.4f}, dur {dur_l.item():.4f})")
    assert expanded.size(1) == target_poses.size(1), "Teacher forcing length mismatch!"
    break


# **13. Inference Pipeline**

One function used by validation, final evaluation, visualization and video
rendering — so those can never drift apart.

In [ ]:
@torch.no_grad()
def generate_poses(input_ids, attention_mask, signer_ids,
                   text_encoder, duration_predictor, length_regulator,
                   temporal_pos_encoder, diffusion_g2p, pose_refiner,
                   max_frames=cfg.MAX_GEN_FRAMES, num_steps=cfg.SAMPLING_STEPS,
                   override_signer=None, guidance=None,
                   smooth_lr=0.0, smooth_steps=20, smooth_iters=3):
    text_encoder.eval(); duration_predictor.eval()
    diffusion_g2p.eval(); pose_refiner.eval()

    if override_signer is not None:
        # Cross-signer evaluation: the model has no embedding for an unseen signer,
        # so we generate AS a signer it knows and compare against the unseen
        # signer's ground truth. State this explicitly when reporting the number.
        signer_ids = torch.full_like(signer_ids, int(override_signer))

    enc = text_encoder(input_ids, attention_mask, signer_ids)
    log_dur = duration_predictor(enc, attention_mask)

    pred_dur = torch.clamp(torch.round(torch.expm1(log_dur)), min=0).long()
    pred_dur = pred_dur * attention_mask.long()
    pred_dur = torch.clamp(pred_dur, max=max_frames)
    # If total predicted frames overflow, scale all tokens down proportionally.
    totals = pred_dur.sum(dim=1)
    for i in range(pred_dur.size(0)):
        if totals[i] > max_frames and totals[i] > 0:
            pred_dur[i] = (pred_dur[i].float() * (max_frames / totals[i].item())).long()
        if pred_dur[i].sum() == 0:  # degenerate: emit at least 1 frame
            pred_dur[i, 0] = 1

    expanded = temporal_pos_encoder(length_regulator(enc, pred_dur))
    g = cfg.CFG_SCALE if guidance is None else guidance
    fm = make_frame_mask(pred_dur.sum(dim=1), expanded.size(1), device)
    raw = diffusion_g2p.sample(expanded, num_steps=num_steps, guidance=g,
                               smooth_lr=smooth_lr, smooth_steps=smooth_steps,
                               smooth_iters=smooth_iters, frame_mask=fm)
    return pose_refiner(raw), pred_dur.sum(dim=1)


@torch.no_grad()
def evaluate_generation(loader, generate_fn, max_batches=None, desc="eval"):
    """Shared by all three architectures. `generate_fn(input_ids, attention_mask,
    signer_ids) -> (pred_poses, pred_lengths)` is the only thing that differs
    between them — everything downstream (DTW alignment, PCK, motion ratio,
    length ratio) is identical, which is what makes Section 16c's table a fair
    comparison rather than three differently-measured numbers."""
    recs = []
    total = min(max_batches, len(loader)) if max_batches is not None else len(loader)
    for bi, batch in enumerate(tqdm(loader, desc=desc, total=total)):
        if max_batches is not None and bi >= max_batches:
            break
        input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
            b.to(device) for b in batch
        ]
        pred, pred_lens = generate_fn(input_ids, attention_mask, signer_ids)
        pred_np = pred.float().cpu().numpy()
        tgt_np = target_poses.float().cpu().numpy()

        for i in range(pred_np.shape[0]):
            tl = int(lengths[i].item())
            pl = int(pred_lens[i].item())
            p = pred_np[i, :max(pl, 1)]
            t = tgt_np[i, :max(tl, 1)]
            m = pose_metrics(p, t)
            m['length_ratio'] = pl / max(tl, 1)
            recs.append(m)
    return recs


@torch.no_grad()
def generate_poses_nat_mse(input_ids, attention_mask, signer_ids,
                           max_frames=cfg.MAX_GEN_FRAMES, override_signer=None):
    """NAT + MSE: same duration -> length-regulator pipeline as
    the diffusion model, but one parallel regression pass instead of diffusion
    sampling. Also 'Parallel (Fast)' at inference."""
    text_encoder_nat_mse.eval(); duration_predictor_nat_mse.eval(); nat_mse_head.eval()
    if override_signer is not None:
        signer_ids = torch.full_like(signer_ids, int(override_signer))

    enc = text_encoder_nat_mse(input_ids, attention_mask, signer_ids)
    log_dur = duration_predictor_nat_mse(enc, attention_mask)
    pred_dur = torch.clamp(torch.round(torch.expm1(log_dur)), min=0).long()
    pred_dur = pred_dur * attention_mask.long()
    pred_dur = torch.clamp(pred_dur, max=max_frames)
    totals = pred_dur.sum(dim=1)
    for i in range(pred_dur.size(0)):
        if totals[i] > max_frames and totals[i] > 0:
            pred_dur[i] = (pred_dur[i].float() * (max_frames / totals[i].item())).long()
        if pred_dur[i].sum() == 0:
            pred_dur[i, 0] = 1

    expanded = temporal_pos_encoder(length_regulator(enc, pred_dur))
    frame_mask = make_frame_mask(pred_dur.sum(dim=1), expanded.size(1), device)
    pred = nat_mse_head(expanded, frame_mask)
    return pred, pred_dur.sum(dim=1)


@torch.no_grad()
def generate_poses_progressive(input_ids, attention_mask, signer_ids,
                               max_frames=cfg.MAX_GEN_FRAMES, override_signer=None):
    """Progressive Transformer: autoregressive, frame-by-frame —
    the 'Sequential (Slow)' path in the architecture comparison table."""
    text_encoder_prog.eval(); progressive_decoder.eval()
    if override_signer is not None:
        signer_ids = torch.full_like(signer_ids, int(override_signer))
    enc = text_encoder_prog(input_ids, attention_mask, signer_ids)
    pose_seq, _, lengths = progressive_decoder.generate(
        enc, cond_mask=attention_mask, max_frames=max_frames)
    return pose_refiner(pose_seq), lengths


@torch.no_grad()
def validation_loss(loader, text_encoder, duration_predictor, length_regulator,
                    temporal_pos_encoder, diffusion_g2p, loss_fn):
    """Cheap teacher-forced loss — run every epoch (no DDIM sampling)."""
    text_encoder.eval(); duration_predictor.eval(); diffusion_g2p.eval()
    tot, n = 0.0, 0
    for batch in loader:
        input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
            b.to(device) for b in batch
        ]
        enc = text_encoder(input_ids, attention_mask, signer_ids)
        log_dur = duration_predictor(enc, attention_mask)
        tgt_dur = compute_uniform_target_durations(attention_mask, lengths)
        expanded = temporal_pos_encoder(length_regulator(enc, tgt_dur))
        pred = diffusion_g2p(target_poses, expanded)
        fm = make_frame_mask(lengths, target_poses.size(1), device)
        l, _, _, _, _ = loss_fn(pred, target_poses, fm, lengths,
                                log_dur, tgt_dur, attention_mask)
        tot += l.item(); n += 1
    return tot / max(n, 1)


@torch.no_grad()
def validation_loss_nat_mse(loader):
    text_encoder_nat_mse.eval(); duration_predictor_nat_mse.eval(); nat_mse_head.eval()
    tot, n = 0.0, 0
    for batch in loader:
        input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
            b.to(device) for b in batch
        ]
        enc = text_encoder_nat_mse(input_ids, attention_mask, signer_ids)
        log_dur = duration_predictor_nat_mse(enc, attention_mask)
        tgt_dur = compute_uniform_target_durations(attention_mask, lengths)
        expanded = temporal_pos_encoder(length_regulator(enc, tgt_dur))
        fm = make_frame_mask(lengths, target_poses.size(1), device)
        pred = nat_mse_head(expanded, fm)
        l, _, _ = mse_loss_fn(pred, target_poses, fm, log_dur, tgt_dur, attention_mask)
        tot += l.item(); n += 1
    return tot / max(n, 1)


@torch.no_grad()
def validation_loss_progressive(loader):
    text_encoder_prog.eval(); progressive_decoder.eval()
    tot, n = 0.0, 0
    for batch in loader:
        input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
            b.to(device) for b in batch
        ]
        enc = text_encoder_prog(input_ids, attention_mask, signer_ids)
        ctr_tgt = compute_counter_targets(lengths, target_poses.size(1), device)
        pred_pose, pred_ctr = progressive_decoder(target_poses, ctr_tgt, enc, cond_mask=attention_mask)
        fm = make_frame_mask(lengths, target_poses.size(1), device)
        l, _, _ = progressive_loss_fn(pred_pose, pred_ctr, target_poses, ctr_tgt, fm)
        tot += l.item(); n += 1
    return tot / max(n, 1)


# **14. Training Loop**

Differential learning rates (BanglaBERT 2e-5 / new layers 1e-4), linear warmup
into cosine decay, AMP for throughput with the loss kept in fp32, per-epoch
validation, early stopping, and separate `best` / `last` checkpoints.

The best-validation checkpoint is the one evaluated and reported: selection uses validation loss only and the test set stays untouched.

In [ ]:
import time
import torch.optim as optim

set_seed(cfg.SEED)

param_groups = [
    {'params': text_encoder.bert_parameters(), 'lr': cfg.BERT_LR},
    {'params': text_encoder.non_bert_parameters(), 'lr': cfg.LR},
    {'params': list(duration_predictor.parameters()), 'lr': cfg.LR},
    {'params': list(diffusion_g2p.parameters()), 'lr': cfg.LR},
]
optimizer = optim.AdamW(param_groups, weight_decay=cfg.WEIGHT_DECAY)
all_params = [p for g in param_groups for p in g['params']]


def lr_lambda(epoch):
    if epoch < cfg.WARMUP_EPOCHS:
        return (epoch + 1) / max(cfg.WARMUP_EPOCHS, 1)
    prog = (epoch - cfg.WARMUP_EPOCHS) / max(cfg.EPOCHS - cfg.WARMUP_EPOCHS, 1)
    return 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))


scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# AMP only makes sense on CUDA; guard so this notebook still runs on CPU.
AMP_ON = bool(cfg.USE_AMP and device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ON)
print(f"Mixed precision (AMP): {AMP_ON}")

best_val = float('inf')
val_ema = None
epochs_no_improve = 0
history = []
start_epoch = 0

BEST_PATH = os.path.join(cfg.CHECKPOINT_DIR, "g2p_best.pt")
LAST_PATH = os.path.join(cfg.CHECKPOINT_DIR, "g2p_last.pt")


def save_ckpt(path, epoch, val_loss, full_state):
    """full_state=False -> weights only (~0.5 GB), enough for evaluation.
    full_state=True -> + optimizer/scheduler/scaler (~1.4 GB), needed to resume."""
    blob = {
        'epoch': epoch,
        'config': {k: v for k, v in vars(Config).items() if not k.startswith('__')},
        'text_encoder_state_dict': text_encoder.state_dict(),
        'duration_state_dict': duration_predictor.state_dict(),
        'diffusion_state_dict': diffusion_g2p.state_dict(),
        'val_loss': val_loss,
        'best_val': best_val,
        'epochs_no_improve': epochs_no_improve,
        'history': history,
    }
    if full_state:
        blob['optimizer_state_dict'] = optimizer.state_dict()
        blob['scheduler_state_dict'] = scheduler.state_dict()
        blob['scaler_state_dict'] = scaler.state_dict()
    torch.save(blob, path)


# ---- Resume: a Kaggle session can die at hour 7 of an 8-hour run. ----
if cfg.RESUME and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device, weights_only=False)
    text_encoder.load_state_dict(ck['text_encoder_state_dict'])
    duration_predictor.load_state_dict(ck['duration_state_dict'])
    diffusion_g2p.load_state_dict(ck['diffusion_state_dict'])
    if 'optimizer_state_dict' in ck:
        optimizer.load_state_dict(ck['optimizer_state_dict'])
    if 'scheduler_state_dict' in ck:
        scheduler.load_state_dict(ck['scheduler_state_dict'])
    if 'scaler_state_dict' in ck:
        scaler.load_state_dict(ck['scaler_state_dict'])
    start_epoch = ck['epoch'] + 1
    best_val = ck.get('best_val', ck.get('val_loss', float('inf')))
    epochs_no_improve = ck.get('epochs_no_improve', 0)
    history = ck.get('history', [])
    print(f"RESUMED from {LAST_PATH} at epoch {start_epoch} (best val {best_val:.4f})")
    print("Set cfg.RESUME=False to start fresh instead.\n")
else:
    print("Starting from scratch.\n")

t_start = time.time()
stop_reason = "completed all epochs"

print(f"Training epochs {start_epoch+1}..{cfg.EPOCHS} | "
      f"patience {cfg.EARLY_STOP_PATIENCE} | time budget {cfg.TIME_BUDGET_HOURS} h\n")

for epoch in range(start_epoch, cfg.EPOCHS if cfg.TRAIN_DIFFUSION else start_epoch):
    if train_sampler is not None:
        train_sampler.set_epoch(epoch)   # fresh batch composition each epoch
    text_encoder.train(); duration_predictor.train(); diffusion_g2p.train()
    run_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.EPOCHS}")

    for batch in pbar:
        input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
            b.to(device, non_blocking=True) for b in batch
        ]
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP_ON):
            enc = text_encoder(input_ids, attention_mask, signer_ids)
            log_dur = duration_predictor(enc, attention_mask)
            tgt_dur = compute_uniform_target_durations(attention_mask, lengths)
            expanded = temporal_pos_encoder(length_regulator(enc, tgt_dur))
            pred = diffusion_g2p(target_poses, expanded)

        # Loss in fp32: Soft-DTW is numerically unstable in fp16.
        fm = make_frame_mask(lengths, target_poses.size(1), device)
        total, dtw_l, bone_l, dur_l, vel_l = loss_fn(
            pred.float(), target_poses.float(), fm, lengths,
            log_dur.float(), tgt_dur, attention_mask
        )

        scaler.scale(total).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(all_params, cfg.GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        run_loss += total.item()
        pbar.set_postfix({"loss": f"{total.item():.4f}", "dtw": f"{dtw_l.item():.4f}",
                          "bone": f"{bone_l.item():.4f}", "vel": f"{vel_l.item():.4f}"})

    scheduler.step()
    train_loss = run_loss / max(len(train_loader), 1)
    val_loss = validation_loss(val_loader, text_encoder, duration_predictor,
                               length_regulator, temporal_pos_encoder,
                               diffusion_g2p, loss_fn)
    history.append({'epoch': epoch + 1, 'train': train_loss, 'val': val_loss})

    elapsed = (time.time() - t_start) / 3600
    per_epoch = elapsed / max(epoch - start_epoch + 1, 1)
    print(f"Epoch {epoch+1:3d} | train {train_loss:.4f} | val {val_loss:.4f} | "
          f"lr {scheduler.get_last_lr()[-1]:.2e} | "
          f"{elapsed:.2f}h elapsed, ~{per_epoch*60:.1f} min/epoch")

    val_ema = val_loss if val_ema is None else cfg.VAL_EMA * val_ema + (1 - cfg.VAL_EMA) * val_loss
    if val_ema < best_val - 1e-5:
        best_val = val_ema
        epochs_no_improve = 0
        save_ckpt(BEST_PATH, epoch, val_loss, full_state=False)
        print(f"  -> new best (val {val_loss:.4f}), saved {BEST_PATH}")
    else:
        epochs_no_improve += 1

    # Full resume state periodically, and always on the final epoch.
    if (epoch + 1) % cfg.SAVE_LAST_EVERY == 0 or epoch == cfg.EPOCHS - 1:
        save_ckpt(LAST_PATH, epoch, val_loss, full_state=True)

    # Stop before Kaggle's 12 h cutoff so the run ends with a usable checkpoint.
    if elapsed + per_epoch > cfg.TIME_BUDGET_HOURS:
        save_ckpt(LAST_PATH, epoch, val_loss, full_state=True)  # flush before stopping
        stop_reason = (f"hit time budget ({cfg.TIME_BUDGET_HOURS} h) at epoch {epoch+1}; "
                       f"re-run this cell with RESUME=True to continue")
        print(f"\n{stop_reason}")
        break

    if (epoch + 1) % cfg.EVAL_EVERY == 0:
        _gen_fn = lambda ii, am, sid: generate_poses(
            ii, am, sid, text_encoder, duration_predictor, length_regulator,
            temporal_pos_encoder, diffusion_g2p, pose_refiner, num_steps=20)
        recs = evaluate_generation(val_loader, _gen_fn,
                                   max_batches=cfg.EVAL_SUBSET_BATCHES,
                                   desc=f"val-gen e{epoch+1}")
        print_metrics(f"VAL GENERATION @ epoch {epoch+1} (subset, 20 steps)",
                      summarize_metrics(recs), len(recs))

    if epochs_no_improve >= cfg.EARLY_STOP_PATIENCE:
        stop_reason = (f"early stopped at epoch {epoch+1} "
                       f"({cfg.EARLY_STOP_PATIENCE} epochs without improvement)")
        print(f"\n{stop_reason}")
        break

print(f"\nTraining finished — {stop_reason}.")
print(f"Best val loss: {best_val:.4f}  ->  {BEST_PATH}")
print(f"Total time: {(time.time()-t_start)/3600:.2f} h")


# **14b. Progressive Transformer Training**

Same data loaders, same differential LR scheme, same warmup/cosine schedule,
same gradient clipping, same early-stopping rule as Section 14 — the only
things that differ are the model and the loss. That is what makes the
Section 16c comparison a controlled one rather than three runs with three
different training regimes.

In [ ]:
set_seed(cfg.SEED)

prog_param_groups = [
    {'params': text_encoder_prog.bert_parameters(), 'lr': cfg.BERT_LR},
    {'params': text_encoder_prog.non_bert_parameters(), 'lr': cfg.LR},
    {'params': list(progressive_decoder.parameters()), 'lr': cfg.LR},
]
prog_optimizer = optim.AdamW(prog_param_groups, weight_decay=cfg.WEIGHT_DECAY)
prog_all_params = [p for g in prog_param_groups for p in g['params']]
prog_scheduler = optim.lr_scheduler.LambdaLR(prog_optimizer, lr_lambda)
prog_scaler = torch.cuda.amp.GradScaler(enabled=AMP_ON)

PROG_BEST_PATH = os.path.join(cfg.CHECKPOINT_DIR, "g2p_progressive_best.pt")
PROG_LAST_PATH = os.path.join(cfg.CHECKPOINT_DIR, "g2p_progressive_last.pt")


def save_ckpt_progressive(path, epoch, val_loss, full_state):
    blob = {
        'epoch': epoch, 'val_loss': val_loss, 'best_val': prog_best_val,
        'epochs_no_improve': prog_epochs_no_improve, 'history': history_progressive,
        'text_encoder_state_dict': text_encoder_prog.state_dict(),
        'decoder_state_dict': progressive_decoder.state_dict(),
    }
    if full_state:
        blob['optimizer_state_dict'] = prog_optimizer.state_dict()
        blob['scheduler_state_dict'] = prog_scheduler.state_dict()
        blob['scaler_state_dict'] = prog_scaler.state_dict()
    torch.save(blob, path)


prog_best_val = float('inf')
prog_val_ema = None
prog_epochs_no_improve = 0
history_progressive = []
prog_start_epoch = 0

if cfg.RESUME and os.path.exists(PROG_LAST_PATH):
    ck = torch.load(PROG_LAST_PATH, map_location=device, weights_only=False)
    text_encoder_prog.load_state_dict(ck['text_encoder_state_dict'])
    progressive_decoder.load_state_dict(ck['decoder_state_dict'])
    if 'optimizer_state_dict' in ck:
        prog_optimizer.load_state_dict(ck['optimizer_state_dict'])
    if 'scheduler_state_dict' in ck:
        prog_scheduler.load_state_dict(ck['scheduler_state_dict'])
    if 'scaler_state_dict' in ck:
        prog_scaler.load_state_dict(ck['scaler_state_dict'])
    prog_start_epoch = ck['epoch'] + 1
    prog_best_val = ck.get('best_val', float('inf'))
    prog_epochs_no_improve = ck.get('epochs_no_improve', 0)
    history_progressive = ck.get('history', [])
    print(f"RESUMED Progressive Transformer from epoch {prog_start_epoch}\n")
else:
    print("Progressive Transformer: starting from scratch.\n")

t_start_prog = time.time()
prog_stop_reason = "completed all epochs"

for epoch in range(prog_start_epoch, cfg.EPOCHS if cfg.TRAIN_PROGRESSIVE else prog_start_epoch):
    if train_sampler is not None:
        train_sampler.set_epoch(epoch)
    text_encoder_prog.train(); progressive_decoder.train()
    run_loss = 0.0
    pbar = tqdm(train_loader, desc=f"[Progressive] Epoch {epoch+1}/{cfg.EPOCHS}")

    for batch in pbar:
        input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
            b.to(device, non_blocking=True) for b in batch
        ]
        prog_optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP_ON):
            enc = text_encoder_prog(input_ids, attention_mask, signer_ids)
            ctr_tgt = compute_counter_targets(lengths, target_poses.size(1), device)
            pred_pose, pred_ctr = progressive_decoder(target_poses, ctr_tgt, enc,
                                                       cond_mask=attention_mask)
        fm = make_frame_mask(lengths, target_poses.size(1), device)
        total, pose_mse, ctr_mse = progressive_loss_fn(
            pred_pose.float(), pred_ctr.float(), target_poses.float(), ctr_tgt.float(), fm)

        prog_scaler.scale(total).backward()
        prog_scaler.unscale_(prog_optimizer)
        torch.nn.utils.clip_grad_norm_(prog_all_params, cfg.GRAD_CLIP_NORM)
        prog_scaler.step(prog_optimizer)
        prog_scaler.update()

        run_loss += total.item()
        pbar.set_postfix({"loss": f"{total.item():.4f}", "pose_mse": f"{pose_mse.item():.4f}",
                          "ctr_mse": f"{ctr_mse.item():.4f}"})

    prog_scheduler.step()
    train_loss = run_loss / max(len(train_loader), 1)
    val_loss = validation_loss_progressive(val_loader)
    history_progressive.append({'epoch': epoch + 1, 'train': train_loss, 'val': val_loss})

    elapsed = (time.time() - t_start_prog) / 3600
    per_epoch = elapsed / max(epoch - prog_start_epoch + 1, 1)
    print(f"[Progressive] Epoch {epoch+1:3d} | train {train_loss:.4f} | val {val_loss:.4f} | "
         f"{elapsed:.2f}h elapsed, ~{per_epoch*60:.1f} min/epoch")

    prog_val_ema = val_loss if prog_val_ema is None else \
        cfg.VAL_EMA * prog_val_ema + (1 - cfg.VAL_EMA) * val_loss
    if prog_val_ema < prog_best_val - 1e-5:
        prog_best_val = prog_val_ema
        prog_epochs_no_improve = 0
        save_ckpt_progressive(PROG_BEST_PATH, epoch, val_loss, full_state=False)
        print(f"  -> new best (val {val_loss:.4f}), saved {PROG_BEST_PATH}")
    else:
        prog_epochs_no_improve += 1

    if (epoch + 1) % cfg.SAVE_LAST_EVERY == 0 or epoch == cfg.EPOCHS - 1:
        save_ckpt_progressive(PROG_LAST_PATH, epoch, val_loss, full_state=True)

    if elapsed + per_epoch > cfg.COMPARISON_TIME_BUDGET_HOURS:
        save_ckpt_progressive(PROG_LAST_PATH, epoch, val_loss, full_state=True)
        prog_stop_reason = (f"hit time budget ({cfg.COMPARISON_TIME_BUDGET_HOURS} h) at "
                            f"epoch {epoch+1}; re-run with RESUME=True to continue")
        print(f"\n{prog_stop_reason}")
        break

    if prog_epochs_no_improve >= cfg.EARLY_STOP_PATIENCE:
        prog_stop_reason = f"early stopped at epoch {epoch+1}"
        print(f"\n{prog_stop_reason}")
        break

print(f"\n[Progressive] Training finished — {prog_stop_reason}.")
print(f"Best val loss: {prog_best_val:.4f}  ->  {PROG_BEST_PATH}")
print(f"Total time: {(time.time()-t_start_prog)/3600:.2f} h")


# **14c. NAT + MSE Training**

In [ ]:
set_seed(cfg.SEED)

nat_param_groups = [
    {'params': text_encoder_nat_mse.bert_parameters(), 'lr': cfg.BERT_LR},
    {'params': text_encoder_nat_mse.non_bert_parameters(), 'lr': cfg.LR},
    {'params': list(duration_predictor_nat_mse.parameters()), 'lr': cfg.LR},
    {'params': list(nat_mse_head.parameters()), 'lr': cfg.LR},
]
nat_optimizer = optim.AdamW(nat_param_groups, weight_decay=cfg.WEIGHT_DECAY)
nat_all_params = [p for g in nat_param_groups for p in g['params']]
nat_scheduler = optim.lr_scheduler.LambdaLR(nat_optimizer, lr_lambda)
nat_scaler = torch.cuda.amp.GradScaler(enabled=AMP_ON)

NAT_MSE_BEST_PATH = os.path.join(cfg.CHECKPOINT_DIR, "g2p_nat_mse_best.pt")
NAT_MSE_LAST_PATH = os.path.join(cfg.CHECKPOINT_DIR, "g2p_nat_mse_last.pt")


def save_ckpt_nat_mse(path, epoch, val_loss, full_state):
    blob = {
        'epoch': epoch, 'val_loss': val_loss, 'best_val': nat_best_val,
        'epochs_no_improve': nat_epochs_no_improve, 'history': history_nat_mse,
        'text_encoder_state_dict': text_encoder_nat_mse.state_dict(),
        'duration_state_dict': duration_predictor_nat_mse.state_dict(),
        'head_state_dict': nat_mse_head.state_dict(),
    }
    if full_state:
        blob['optimizer_state_dict'] = nat_optimizer.state_dict()
        blob['scheduler_state_dict'] = nat_scheduler.state_dict()
        blob['scaler_state_dict'] = nat_scaler.state_dict()
    torch.save(blob, path)


nat_best_val = float('inf')
nat_val_ema = None
nat_epochs_no_improve = 0
history_nat_mse = []
nat_start_epoch = 0

if cfg.RESUME and os.path.exists(NAT_MSE_LAST_PATH):
    ck = torch.load(NAT_MSE_LAST_PATH, map_location=device, weights_only=False)
    text_encoder_nat_mse.load_state_dict(ck['text_encoder_state_dict'])
    duration_predictor_nat_mse.load_state_dict(ck['duration_state_dict'])
    nat_mse_head.load_state_dict(ck['head_state_dict'])
    if 'optimizer_state_dict' in ck:
        nat_optimizer.load_state_dict(ck['optimizer_state_dict'])
    if 'scheduler_state_dict' in ck:
        nat_scheduler.load_state_dict(ck['scheduler_state_dict'])
    if 'scaler_state_dict' in ck:
        nat_scaler.load_state_dict(ck['scaler_state_dict'])
    nat_start_epoch = ck['epoch'] + 1
    nat_best_val = ck.get('best_val', float('inf'))
    nat_epochs_no_improve = ck.get('epochs_no_improve', 0)
    history_nat_mse = ck.get('history', [])
    print(f"RESUMED NAT+MSE from epoch {nat_start_epoch}\n")
else:
    print("NAT + MSE: starting from scratch.\n")

t_start_nat = time.time()
nat_stop_reason = "completed all epochs"

for epoch in range(nat_start_epoch, cfg.EPOCHS):
    if train_sampler is not None:
        train_sampler.set_epoch(epoch)
    text_encoder_nat_mse.train(); duration_predictor_nat_mse.train(); nat_mse_head.train()
    run_loss = 0.0
    pbar = tqdm(train_loader, desc=f"[NAT+MSE] Epoch {epoch+1}/{cfg.EPOCHS}")

    for batch in pbar:
        input_ids, attention_mask, target_poses, lengths, signer_ids, _, _ = [
            b.to(device, non_blocking=True) for b in batch
        ]
        nat_optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP_ON):
            enc = text_encoder_nat_mse(input_ids, attention_mask, signer_ids)
            log_dur = duration_predictor_nat_mse(enc, attention_mask)
            tgt_dur = compute_uniform_target_durations(attention_mask, lengths)
            expanded = temporal_pos_encoder(length_regulator(enc, tgt_dur))
            fm = make_frame_mask(lengths, target_poses.size(1), device)
            pred = nat_mse_head(expanded, fm)

        total, mse, dur_l = mse_loss_fn(pred.float(), target_poses.float(), fm,
                                        log_dur.float(), tgt_dur, attention_mask)

        nat_scaler.scale(total).backward()
        nat_scaler.unscale_(nat_optimizer)
        torch.nn.utils.clip_grad_norm_(nat_all_params, cfg.GRAD_CLIP_NORM)
        nat_scaler.step(nat_optimizer)
        nat_scaler.update()

        run_loss += total.item()
        pbar.set_postfix({"loss": f"{total.item():.4f}", "mse": f"{mse.item():.4f}",
                          "dur": f"{dur_l.item():.4f}"})

    nat_scheduler.step()
    train_loss = run_loss / max(len(train_loader), 1)
    val_loss = validation_loss_nat_mse(val_loader)
    history_nat_mse.append({'epoch': epoch + 1, 'train': train_loss, 'val': val_loss})

    elapsed = (time.time() - t_start_nat) / 3600
    per_epoch = elapsed / max(epoch - nat_start_epoch + 1, 1)
    print(f"[NAT+MSE] Epoch {epoch+1:3d} | train {train_loss:.4f} | val {val_loss:.4f} | "
         f"{elapsed:.2f}h elapsed, ~{per_epoch*60:.1f} min/epoch")

    nat_val_ema = val_loss if nat_val_ema is None else \
        cfg.VAL_EMA * nat_val_ema + (1 - cfg.VAL_EMA) * val_loss
    if nat_val_ema < nat_best_val - 1e-5:
        nat_best_val = nat_val_ema
        nat_epochs_no_improve = 0
        save_ckpt_nat_mse(NAT_MSE_BEST_PATH, epoch, val_loss, full_state=False)
        print(f"  -> new best (val {val_loss:.4f}), saved {NAT_MSE_BEST_PATH}")
    else:
        nat_epochs_no_improve += 1

    if (epoch + 1) % cfg.SAVE_LAST_EVERY == 0 or epoch == cfg.EPOCHS - 1:
        save_ckpt_nat_mse(NAT_MSE_LAST_PATH, epoch, val_loss, full_state=True)

    if elapsed + per_epoch > cfg.COMPARISON_TIME_BUDGET_HOURS:
        save_ckpt_nat_mse(NAT_MSE_LAST_PATH, epoch, val_loss, full_state=True)
        nat_stop_reason = (f"hit time budget ({cfg.COMPARISON_TIME_BUDGET_HOURS} h) at "
                           f"epoch {epoch+1}; re-run with RESUME=True to continue")
        print(f"\n{nat_stop_reason}")
        break

    if nat_epochs_no_improve >= cfg.EARLY_STOP_PATIENCE:
        nat_stop_reason = f"early stopped at epoch {epoch+1}"
        print(f"\n{nat_stop_reason}")
        break

print(f"\n[NAT+MSE] Training finished — {nat_stop_reason}.")
print(f"Best val loss: {nat_best_val:.4f}  ->  {NAT_MSE_BEST_PATH}")
print(f"Total time: {(time.time()-t_start_nat)/3600:.2f} h")


# **15. Training Curves**

In [ ]:
import matplotlib.pyplot as plt

ep = [h['epoch'] for h in history]
tr = [h['train'] for h in history]
va = [h['val'] for h in history]

plt.figure(figsize=(9, 4))
plt.plot(ep, tr, label='train')
plt.plot(ep, va, label='val')
if va:
    plt.axvline(int(np.argmin(va)) + 1, ls='--', c='gray', label='best val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend()
plt.title('Training / validation loss (length-normalized)')
plt.tight_layout(); plt.show()

import csv
hist_path = os.path.join(cfg.CHECKPOINT_DIR, "history.csv")
with open(hist_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=['epoch', 'train', 'val'])
    w.writeheader(); w.writerows(history)
print(f"History written to {hist_path}")


# **15b. Combined Training Curves — Three Architectures**

Training and validation curves for all three models. Each model's loss lives on a different scale — plain MSE, a masked MSE +
counter MSE, and a length-normalized Soft-DTW + bone + velocity + duration
stack are not numerically comparable — so this is three subplots sharing an
epoch axis rather than one shared y-axis. What *is* comparable across panels:
**shape**. Does validation loss ever turn upward (overfitting)? Does it
plateau early (underfitting / architecture ceiling) or late? Does the gap
between train and val stay small or widen? Those shape comparisons, not the raw loss values, are what this figure supports.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharex=False)

panels = [
    ("Progressive Transformer", history_progressive, axes[0]),
    ("NAT + MSE", history_nat_mse, axes[1]),
    ("NAT + Diffusion + Velocity", history, axes[2]),
]

for title, hist, ax in panels:
    if not hist:
        ax.set_title(f"{title}\n(not trained in this session)")
        continue
    ep = [h['epoch'] for h in hist]
    tr = [h['train'] for h in hist]
    va = [h['val'] for h in hist]
    ax.plot(ep, tr, label='train')
    ax.plot(ep, va, label='val')
    ax.axvline(int(np.argmin(va)) + 1, ls='--', c='gray', lw=1, label='best val')
    ax.set_xlabel('epoch'); ax.set_ylabel('loss')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("Training / validation loss by architecture (not cross-comparable in magnitude — see note above)")
plt.tight_layout()
plt.show()

for tag, hist in [("progressive", history_progressive), ("nat_mse", history_nat_mse)]:
    if not hist:
        continue
    p = os.path.join(cfg.CHECKPOINT_DIR, f"history_{tag}.csv")
    with open(p, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=['epoch', 'train', 'val'])
        w.writeheader(); w.writerows(hist)
    print(f"History written to {p}")


# **16. Final Test-Set Evaluation**

Loads `g2p_best.pt` (selected on validation) and evaluates on the **test set**,
which was never used for training or model selection. These are the reported test numbers.

In [ ]:
import csv

print(f"Loading best checkpoint: {BEST_PATH}")
ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
text_encoder.load_state_dict(ckpt['text_encoder_state_dict'])
duration_predictor.load_state_dict(ckpt['duration_state_dict'])
diffusion_g2p.load_state_dict(ckpt['diffusion_state_dict'])
print(f"  (from epoch {ckpt['epoch']+1}, val loss {ckpt['val_loss']:.4f})")

_gen_fn_diffusion = lambda ii, am, sid: generate_poses(
    ii, am, sid, text_encoder, duration_predictor, length_regulator,
    temporal_pos_encoder, diffusion_g2p, pose_refiner)
test_records = evaluate_generation(test_loader, _gen_fn_diffusion, desc="TEST")
test_metrics = summarize_metrics(test_records)
print_metrics(f"TEST SET — {cfg.SPLIT_MODE} split", test_metrics, len(test_records))

# Per-sample values saved for significance testing across seeds .
res_path = os.path.join(cfg.CHECKPOINT_DIR, f"test_results_seed{cfg.SEED}.csv")
with open(res_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(test_records[0].keys()))
    w.writeheader(); w.writerows(test_records)
print(f"\nPer-sample metrics -> {res_path}")
print("Keep these files: comparing seeds/ablations needs per-sample values, not just means.")


# **16b. Cross-Signer Generalization**

When training uses a subset of signers, the same held-out sentences signed by the other signers
form a ready-made generalization test that needs no extra training, only another sampling pass.

The model has no embedding for a signer it never saw, so it generates as a signer it knows and is
scored against the unseen signer's recording. Report this as "generated as signer *k*, evaluated
against signer *j*". Because poses are normalized, body size and camera position are factored out
and what remains is signing-style difference; a gap is expected and is not by itself evidence of
overfitting or of robustness.


In [ ]:
if cross_loader is None:
    print("Cross-signer eval skipped (SIGNERS_TO_USE is None, so no signer was held out).")
elif not cfg.CROSS_SIGNER_EVAL:
    print("Cross-signer eval disabled via cfg.CROSS_SIGNER_EVAL.")
else:
    _gen_fn_cross = lambda ii, am, sid: generate_poses(
        ii, am, sid, text_encoder, duration_predictor, length_regulator,
        temporal_pos_encoder, diffusion_g2p, pose_refiner, override_signer=PRIMARY_SIGNER)
    cross_records = evaluate_generation(cross_loader, _gen_fn_cross, desc="CROSS-SIGNER")
    cross_metrics = summarize_metrics(cross_records)
    print_metrics(f"CROSS-SIGNER TEST — generated as signer {PRIMARY_SIGNER}, "
                  f"scored vs unseen signers", cross_metrics, len(cross_records))

    print("\n--- Same-signer vs cross-signer ---")
    d_in = test_metrics.get('dtw_mje', float('nan'))
    d_out = cross_metrics.get('dtw_mje', float('nan'))
    print(f"DTW-MJE  same-signer {d_in:.4f}  |  cross-signer {d_out:.4f}  "
          f"|  gap {d_out - d_in:+.4f} ({100*(d_out-d_in)/max(d_in,1e-9):+.1f}%)")
    for t in PCK_THRESHOLDS:
        k = f"PCK@{t}"
        print(f"{k:<9} same-signer {test_metrics.get(k, float('nan')):6.2f}%  |  "
              f"cross-signer {cross_metrics.get(k, float('nan')):6.2f}%")

    cross_path = os.path.join(cfg.CHECKPOINT_DIR, f"cross_signer_seed{cfg.SEED}.csv")
    with open(cross_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(cross_records[0].keys()))
        w.writeheader(); w.writerows(cross_records)
    print(f"\nPer-sample metrics -> {cross_path}")


# **16e. Classifier-Free Guidance Sweep (velocity / guidance ablation)**

Guidance acts only at sampling time, so one trained checkpoint can be scored
at several `CFG_SCALE` values for the cost of a few evaluation passes. Pair this
with runs at different `VELOCITY_WEIGHT` to separate the two suspected causes of
over-motion: the velocity term (a training-time effect) and guidance (a
sampling-time effect). Only runs when `RUN_CFG_SWEEP` is True.

In [ ]:
if not cfg.RUN_CFG_SWEEP:
    print("CFG sweep disabled (cfg.RUN_CFG_SWEEP is False).")
else:
    import pandas as pd
    rows = []
    for g in cfg.CFG_SWEEP:
        gen = lambda ii, am, sid, g=g: generate_poses(
            ii, am, sid, text_encoder, duration_predictor, length_regulator,
            temporal_pos_encoder, diffusion_g2p, pose_refiner, guidance=g)
        recs = evaluate_generation(test_loader, gen, desc=f"CFG {g}")
        m = summarize_metrics(recs)
        print_metrics(f"velocity_weight={cfg.VELOCITY_WEIGHT:g}  cfg_scale={g:g}", m, len(recs))
        rows.append({
            'velocity_weight': cfg.VELOCITY_WEIGHT, 'cfg_scale': g,
            'epochs_trained': len(history), 'best_epoch': int(ckpt['epoch']) + 1,
            'motion_ratio': m['motion_ratio'], 'jitter_ratio': m['jitter_ratio'],
            'dtw_mje': m['dtw_mje'], 'pck_0.05': m['PCK@0.05'], 'pck_0.1': m['PCK@0.1'],
            'pck_0.2': m['PCK@0.2'], 'pck_0.5': m['PCK@0.5'],
            'length_ratio': m['length_ratio'], 'n_test_videos': len(recs),
        })
        with open(os.path.join(cfg.CHECKPOINT_DIR,
                  f"test_results_vel{cfg.VELOCITY_WEIGHT:g}_cfg{g:g}_seed{cfg.SEED}.csv"),
                  "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(recs[0].keys()))
            w.writeheader(); w.writerows(recs)

    sweep_df = pd.DataFrame(rows)
    print("\n" + "=" * 100)
    print(f" CFG SWEEP — velocity_weight={cfg.VELOCITY_WEIGHT:g}, best checkpoint, same test set")
    print("=" * 100)
    print(sweep_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    sweep_path = os.path.join(cfg.CHECKPOINT_DIR, f"cfg_sweep_vel{cfg.VELOCITY_WEIGHT:g}.csv")
    sweep_df.to_csv(sweep_path, index=False)
    print(f"\nSaved -> {sweep_path}")


# **16f. Test-Time Smoothness Guidance vs Plain Smoothing**

The velocity x guidance ablation found the diffusion model's output has a jitter
ratio of 2-4x ground truth. MotionFlow (Meral et al., AAAI 2026) fixes a related
problem in video diffusion without training: during the first sampling steps it
nudges the latents by the gradient of a temporal-consistency loss. Here the loss
is the squared frame-to-frame acceleration of the predicted pose. The control is
simply averaging the output over a wider window, which needs no gradients: if
guidance does no better than that, it is not worth its cost. Arms are scored on
the validation set first (use that to choose), then on the test set (report it
once). Inference-only; runs only when `RUN_SMOOTH_GUIDANCE` is True.

In [ ]:
if not cfg.RUN_SMOOTH_GUIDANCE:
    print("Smoothness-guidance study disabled (cfg.RUN_SMOOTH_GUIDANCE is False).")
else:
    import pandas as pd
    arms = [
        ("default (7-frame refiner)",     dict(kernel=7,  lr=0.0)),
        ("control: refiner 15",           dict(kernel=15, lr=0.0)),
        ("control: refiner 31",           dict(kernel=31, lr=0.0)),
        ("guidance lr 0.02",              dict(kernel=7,  lr=0.02)),
        ("guidance lr 0.05",              dict(kernel=7,  lr=0.05)),
        ("guidance lr 0.10",              dict(kernel=7,  lr=0.10)),
        ("guidance lr 0.05 + refiner 15", dict(kernel=15, lr=0.05)),
    ]
    rows = []
    for name, a in arms:
        refiner_k = PoseRefiner(kernel_size=a["kernel"]).to(device)
        gen = lambda ii, am, sid, a=a, rk=refiner_k: generate_poses(
            ii, am, sid, text_encoder, duration_predictor, length_regulator,
            temporal_pos_encoder, diffusion_g2p, rk, smooth_lr=a["lr"])
        for split, loader in (("val", val_loader), ("test", test_loader)):
            t0 = time.time()
            recs = evaluate_generation(loader, gen, desc=f"{name} [{split}]")
            m = summarize_metrics(recs)
            rows.append({
                "arm": name, "split": split, "cfg_scale": cfg.CFG_SCALE,
                "motion_ratio": m["motion_ratio"], "jitter_ratio": m["jitter_ratio"],
                "dtw_mje": m["dtw_mje"], "pck_0.2": m["PCK@0.2"], "pck_0.5": m["PCK@0.5"],
                "length_ratio": m["length_ratio"], "n_videos": len(recs),
                "wall_seconds": round(time.time() - t0, 1),
            })
            print(f"{name:32s} {split:4s} DTW-MJE {m['dtw_mje']:.4f}  motion {m['motion_ratio']:.3f}"
                  f"  jitter {m['jitter_ratio']:.3f}  PCK@0.5 {m['PCK@0.5']:.2f}%")
    smooth_df = pd.DataFrame(rows)
    print("\n" + "=" * 100)
    print(smooth_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    out_path = os.path.join(cfg.CHECKPOINT_DIR, "smooth_guidance_arms.csv")
    smooth_df.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")


# **16g. NAT Variant Evaluation (temporal-variance penalty)**

Scores the NAT+MSE-family model trained in Section 14c (best-validation
checkpoint) on the validation set, the test set and the unseen-signer set, with
the same `pose_metrics` pipeline as everything else, including jitter ratio.
`VARIANCE_WEIGHT = 0` is exactly the plain NAT + MSE model. Choose the weight on validation and
quote the test numbers once. Only runs when `RUN_NAT_EVAL` is True.

In [ ]:
if not cfg.RUN_NAT_EVAL:
    print("NAT variant evaluation disabled (cfg.RUN_NAT_EVAL is False).")
else:
    import csv
    import pandas as pd
    _ck = torch.load(NAT_MSE_BEST_PATH, map_location=device, weights_only=False)
    text_encoder_nat_mse.load_state_dict(_ck['text_encoder_state_dict'])
    duration_predictor_nat_mse.load_state_dict(_ck['duration_state_dict'])
    nat_mse_head.load_state_dict(_ck['head_state_dict'])
    rows = []
    for split, loader, ovr in (("val", val_loader, None), ("test", test_loader, None),
                               ("cross_signer", cross_loader, PRIMARY_SIGNER)):
        if loader is None:
            continue
        gen = lambda ii, am, sid, o=ovr: generate_poses_nat_mse(ii, am, sid, override_signer=o)
        recs = evaluate_generation(loader, gen, desc=f"NAT w={cfg.VARIANCE_WEIGHT:g} [{split}]")
        m = summarize_metrics(recs)
        print_metrics(f"NAT variance_weight={cfg.VARIANCE_WEIGHT:g}  [{split}]", m, len(recs))
        rows.append({
            "variance_weight": cfg.VARIANCE_WEIGHT, "split": split,
            "epochs_trained": len(history_nat_mse), "best_epoch": int(_ck["epoch"]) + 1,
            "motion_ratio": m["motion_ratio"], "jitter_ratio": m["jitter_ratio"],
            "dtw_mje": m["dtw_mje"], "pck_0.05": m["PCK@0.05"], "pck_0.1": m["PCK@0.1"],
            "pck_0.2": m["PCK@0.2"], "pck_0.5": m["PCK@0.5"],
            "length_ratio": m["length_ratio"], "n_videos": len(recs),
        })
        if split == "test":
            with open(os.path.join(cfg.CHECKPOINT_DIR,
                      f"test_results_natvar_w{cfg.VARIANCE_WEIGHT:g}_seed{cfg.SEED}.csv"),
                      "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(recs[0].keys()))
                w.writeheader(); w.writerows(recs)
    nat_df = pd.DataFrame(rows)
    print("\n" + "=" * 100)
    print(nat_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    out_path = os.path.join(cfg.CHECKPOINT_DIR, f"nat_variance_w{cfg.VARIANCE_WEIGHT:g}.csv")
    nat_df.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")


# **16h. Back-Translation I - Train a Pose-to-Gloss Recogniser**

DTW-MJE and PCK measure geometry, not meaning. This asks the question that matters for
sign production: can the generated signing be *read back* as the sentence it was
generated from? A CTC transformer is trained on **real training poses only** (all five
signers, so it is not tuned to the one signer the generators imitate; sentence-disjoint
from validation and test, split on gloss text) and early-stopped on real validation
poses. Training augments with smoothing, noise and speed changes so it is not
unfairly harsh on smooth or jittery output.

The corpus has only 29 distinct gloss tokens in a fixed template, so chance token
accuracy is high: Section 16j reports a wrong-sentence floor and a real-pose ceiling
for exactly that reason. Only runs when `RUN_BACKTRANS` is True.

In [ ]:
if not cfg.RUN_BACKTRANS:
    print("Back-translation study disabled (cfg.RUN_BACKTRANS is False).")
else:
    import math
    import torch.nn.functional as F

    def gloss_tokens(g):
        return " ".join(str(g).split()).strip().rstrip("।.").split()

    train_keys = {r["split_key"] for r in train_recs}
    val_keys = {r["split_key"] for r in val_recs}
    rec_train = [r for r in all_records if r["split_key"] in train_keys]   # all signers
    rec_val = [r for r in all_records if r["split_key"] in val_keys]
    vocab = sorted({t for r in rec_train for t in gloss_tokens(r["gloss"])})
    tok2id = {t: i + 1 for i, t in enumerate(vocab)}                       # 0 = CTC blank
    id2tok = {i: t for t, i in tok2id.items()}
    oov = {t for r in all_records for t in gloss_tokens(r["gloss"])} - set(vocab)
    print(f"gloss vocabulary: {len(vocab)} tokens (train only); tokens never seen in training: {len(oov)}")
    print(f"recogniser data: train {len(rec_train)} recordings ({len({r['signer_id'] for r in rec_train})} signers), "
          f"val {len(rec_val)}")

    # Legs (23-32) are extrapolated by the pose extractor, so the recogniser ignores them.
    REC_JOINTS = list(range(0, cfg.LEG_START)) + list(range(cfg.HAND_START, cfg.NUM_JOINTS))

    def load_rec_pose(rec):
        p = torch.tensor(np.load(rec["tensor_path"]), dtype=torch.float32)
        if p.ndim == 2:
            p = p.view(p.shape[0], cfg.NUM_JOINTS, cfg.COORD_DIM)
        p, _, _ = train_ds._normalize_pose(p)
        return p[:, REC_JOINTS].contiguous()

    rec_train_x = [load_rec_pose(r) for r in tqdm(rec_train, desc="cache train poses")]
    rec_val_x = [load_rec_pose(r) for r in tqdm(rec_val, desc="cache val poses")]
    rec_train_y = [[tok2id[t] for t in gloss_tokens(r["gloss"])] for r in rec_train]
    rec_val_ref = [gloss_tokens(r["gloss"]) for r in rec_val]

    def moving_avg(p, k):
        if k <= 1 or p.shape[0] < 3:
            return p
        x = p.reshape(p.shape[0], -1).t().unsqueeze(0)
        x = F.pad(x, (k // 2, k // 2), mode="replicate")
        return F.avg_pool1d(x, k, stride=1).squeeze(0).t().reshape(p.shape)

    def rec_augment(p):
        # Cover the artefacts generated poses can have: smoothing, noise, speed change.
        p = moving_avg(p, random.choice([1, 1, 3, 5, 7, 9, 15]))
        sp = random.uniform(0.85, 1.15)
        if abs(sp - 1.0) > 0.02:
            T = max(8, int(round(p.shape[0] * sp)))
            q = F.interpolate(p.reshape(p.shape[0], -1).t().unsqueeze(0), size=T,
                              mode="linear", align_corners=False)
            p = q.squeeze(0).t().reshape((T,) + tuple(p.shape[1:]))
        return p + random.uniform(0.0, 0.02) * torch.randn_like(p)

    def rec_features(x):
        f = x.reshape(x.shape[0], x.shape[1], -1)
        v = torch.zeros_like(f)
        v[:, 1:] = f[:, 1:] - f[:, :-1]
        return torch.cat([f, 5.0 * v], dim=-1)

    def edit_distance(a, b):
        d = list(range(len(b) + 1))
        for i in range(1, len(a) + 1):
            prev, d[0] = d[0], i
            for j in range(1, len(b) + 1):
                cur = d[j]
                d[j] = min(d[j] + 1, d[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
                prev = cur
        return d[len(b)]

    class PoseGlossRecognizer(nn.Module):
        def __init__(self, n_vocab, d=256, layers=4, heads=8):
            super().__init__()
            in_dim = 2 * len(REC_JOINTS) * cfg.COORD_DIM
            self.inp = nn.Sequential(nn.Linear(in_dim, d), nn.LayerNorm(d), nn.Dropout(0.1))
            self.pos = TemporalPositionalEncoding(d)
            self.enc = nn.TransformerEncoder(
                nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.1, batch_first=True,
                                           norm_first=True),
                layers, norm=nn.LayerNorm(d))
            self.out = nn.Linear(d, n_vocab + 1)

        def forward(self, feats, lengths):
            T = feats.shape[1]
            pad = torch.arange(T, device=feats.device).unsqueeze(0) >= lengths.unsqueeze(1)
            return self.out(self.enc(self.pos(self.inp(feats)), src_key_padding_mask=pad))

    recognizer = PoseGlossRecognizer(len(vocab)).to(device)

    def rec_pad(xs):
        lens = torch.tensor([x.shape[0] for x in xs], device=device)
        return pad_sequence(xs, batch_first=True).to(device), lens

    @torch.no_grad()
    def recognize(xs, bs=64):
        recognizer.eval()
        order = sorted(range(len(xs)), key=lambda i: xs[i].shape[0])
        hyps = [None] * len(xs)
        for s0 in range(0, len(xs), bs):
            idx = order[s0:s0 + bs]
            X, lens = rec_pad([xs[i] for i in idx])
            pred = recognizer(rec_features(X), lens).argmax(-1).cpu()
            for b, i in enumerate(idx):
                out, prev = [], 0
                for t in pred[b, :int(lens[b])].tolist():
                    if t != prev and t != 0:
                        out.append(id2tok[t])
                    prev = t
                hyps[i] = out
        return hyps

    def score_hyps(hyps, refs):
        ed = sum(edit_distance(h, r) for h, r in zip(hyps, refs))
        n = sum(len(r) for r in refs)
        return sum(h == r for h, r in zip(hyps, refs)) / len(refs), ed / max(n, 1)

    REC_PATH = os.path.join(cfg.CHECKPOINT_DIR, "recognizer_best.pt")
    rec_opt = optim.AdamW(recognizer.parameters(), lr=cfg.REC_LR, weight_decay=1e-4)
    # 3-epoch linear warmup, then cosine: CTC models trained from scratch often sit on an
    # all-blank plateau or diverge without it.
    rec_sched = optim.lr_scheduler.LambdaLR(
        rec_opt, lambda e: min(1.0, (e + 1) / 3) * 0.5 * (1 + math.cos(math.pi * e / cfg.REC_EPOCHS)))
    ctc = nn.CTCLoss(blank=0, zero_infinity=True)
    best_val, bad, rec_hist = float("inf"), 0, []
    for ep in range(cfg.REC_EPOCHS):
        recognizer.train()
        perm = list(range(len(rec_train_x)))
        random.shuffle(perm)
        run = 0.0
        for s0 in range(0, len(perm), cfg.REC_BATCH):
            ids = perm[s0:s0 + cfg.REC_BATCH]
            X, lens = rec_pad([rec_augment(rec_train_x[i]) for i in ids])
            ys = [rec_train_y[i] for i in ids]
            tgt = torch.tensor([t for y in ys for t in y], dtype=torch.long)
            lp = recognizer(rec_features(X), lens).float().log_softmax(-1).transpose(0, 1)
            loss = ctc(lp, tgt, lens.cpu(), torch.tensor([len(y) for y in ys]))
            rec_opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(recognizer.parameters(), 1.0)
            rec_opt.step()
            run += loss.item()
        rec_sched.step()
        v_acc, v_wer = score_hyps(recognize(rec_val_x), rec_val_ref)
        rec_hist.append({"epoch": ep + 1, "train_ctc": run / math.ceil(len(perm) / cfg.REC_BATCH),
                         "val_exact": v_acc, "val_wer": v_wer})
        print(f"[recogniser] epoch {ep + 1:3d} | ctc {rec_hist[-1]['train_ctc']:.4f} | "
              f"val exact {100 * v_acc:.1f}% | val WER {100 * v_wer:.1f}%")
        if v_wer < best_val - 1e-6:
            best_val, bad = v_wer, 0
            torch.save({"state_dict": recognizer.state_dict(), "vocab": vocab, "epoch": ep + 1}, REC_PATH)
        else:
            bad += 1
            if bad >= cfg.REC_PATIENCE:
                print(f"[recogniser] early stopped at epoch {ep + 1}")
                break
    recognizer.load_state_dict(torch.load(REC_PATH, map_location=device, weights_only=False)["state_dict"])
    v_acc, v_wer = score_hyps(recognize(rec_val_x), rec_val_ref)
    print(f"recogniser (best checkpoint) on REAL validation poses, all signers: "
          f"exact {100 * v_acc:.1f}%, WER {100 * v_wer:.1f}%")


# **16i. Back-Translation II - Generate and Recognise**

Every model generates the 192 test sentences (single signer), then the recogniser reads
the poses. Conditions: real test poses (ceiling), real poses smoothed (does smoothing
alone hurt recognition?), Progressive Transformer, NAT+MSE, and the diffusion model at
three guidance scales (guidance is the conditioning strength, so it may matter for
meaning even where it did not help geometry).

In [ ]:
if not cfg.RUN_BACKTRANS:
    print("Back-translation study disabled (cfg.RUN_BACKTRANS is False).")
else:
    _c = torch.load(BEST_PATH, map_location=device, weights_only=False)
    text_encoder.load_state_dict(_c["text_encoder_state_dict"])
    duration_predictor.load_state_dict(_c["duration_state_dict"])
    diffusion_g2p.load_state_dict(_c["diffusion_state_dict"])
    _c = torch.load(PROG_BEST_PATH, map_location=device, weights_only=False)
    text_encoder_prog.load_state_dict(_c["text_encoder_state_dict"])
    progressive_decoder.load_state_dict(_c["decoder_state_dict"])
    _c = torch.load(NAT_MSE_BEST_PATH, map_location=device, weights_only=False)
    text_encoder_nat_mse.load_state_dict(_c["text_encoder_state_dict"])
    duration_predictor_nat_mse.load_state_dict(_c["duration_state_dict"])
    nat_mse_head.load_state_dict(_c["head_state_dict"])

    generators = [("Progressive Transformer", lambda ii, am, sid: generate_poses_progressive(ii, am, sid)),
                  ("NAT + MSE", lambda ii, am, sid: generate_poses_nat_mse(ii, am, sid))]
    for g in cfg.BT_CFG_SCALES:
        generators.append((f"Diffusion (CFG {g:g})",
                           lambda ii, am, sid, g=g: generate_poses(
                               ii, am, sid, text_encoder, duration_predictor, length_regulator,
                               temporal_pos_encoder, diffusion_g2p, pose_refiner, guidance=g)))

    # Iterate the test set in an order we control, so every item keeps its reference gloss.
    bt_sampler = LengthGroupedBatchSampler(scan_lengths(test_ds), cfg.BATCH_SIZE,
                                           shuffle=False, seed=cfg.SEED)
    bt_idx_batches = list(bt_sampler)
    bt_loader = DataLoader(test_ds, batch_sampler=bt_sampler, collate_fn=pad_collate_fn,
                           num_workers=cfg.NUM_WORKERS)
    N = len(test_ds)
    bt_refs = [gloss_tokens(test_ds.records[i]["gloss"]) for i in range(N)]
    bt_poses = {"Real test poses": [None] * N}
    for name, _ in generators:
        bt_poses[name] = [None] * N

    for batch, idxs in tqdm(zip(bt_loader, bt_idx_batches), total=len(bt_idx_batches), desc="generate"):
        ii, am, tgt, lens, sid, _, _ = [b.to(device) for b in batch]
        for name, fn in generators:
            pred, plen = fn(ii, am, sid)
            for b, di in enumerate(idxs):
                bt_poses[name][di] = pred[b, :max(int(plen[b]), 1)][:, REC_JOINTS].float().cpu()
        for b, di in enumerate(idxs):
            bt_poses["Real test poses"][di] = tgt[b, :int(lens[b])][:, REC_JOINTS].float().cpu()

    bt_poses["Real test poses, smoothed (7-frame)"] = [moving_avg(p, 7) for p in bt_poses["Real test poses"]]
    bt_poses["Real test poses, smoothed (15-frame)"] = [moving_avg(p, 15) for p in bt_poses["Real test poses"]]
    bt_hyps = {name: recognize(xs) for name, xs in bt_poses.items()}
    print("recognised", {name: len(h) for name, h in bt_hyps.items()})


# **16j. Back-Translation III - Metrics, Controls, Confidence Intervals**

Sentence exact-match accuracy, token WER, BLEU-1/2 (sentences are 3-7 tokens, so
higher-order BLEU is not meaningful), 95% bootstrap CIs resampling sentences, and
paired bootstrap differences between models. Treat the real-pose row as the ceiling
and the wrong-sentence row as the floor; a model is only 'readable' relative to those.

In [ ]:
if not cfg.RUN_BACKTRANS:
    print("Back-translation study disabled (cfg.RUN_BACKTRANS is False).")
else:
    import pandas as pd

    def bleu(refs, hyps, max_n):
        logp = 0.0
        for n in range(1, max_n + 1):
            match = total = 0
            for r, h in zip(refs, hyps):
                rc, hc = {}, {}
                for i in range(len(r) - n + 1):
                    rc[tuple(r[i:i + n])] = rc.get(tuple(r[i:i + n]), 0) + 1
                for i in range(len(h) - n + 1):
                    hc[tuple(h[i:i + n])] = hc.get(tuple(h[i:i + n]), 0) + 1
                match += sum(min(c, rc.get(k, 0)) for k, c in hc.items())
                total += max(len(h) - n + 1, 0)
            logp += math.log(max(match, 1e-9) / max(total, 1))
        hl, rl = sum(len(h) for h in hyps), sum(len(r) for r in refs)
        bp = 1.0 if hl >= rl else math.exp(1 - rl / max(hl, 1))
        return 100 * bp * math.exp(logp / max_n)

    def per_sentence(hyps, refs):
        ed = np.array([edit_distance(h, r) for h, r in zip(hyps, refs)], dtype=float)
        rl = np.array([len(r) for r in refs], dtype=float)
        ex = np.array([h == r for h, r in zip(hyps, refs)], dtype=float)
        return ed, rl, ex

    rows, store = [], {}
    for name, hyps in bt_hyps.items():
        store[name] = per_sentence(hyps, bt_refs)

    # Floor: the same real-pose hypotheses scored against other sentences' glosses.
    rng = np.random.default_rng(0)
    real_h = bt_hyps["Real test poses"]
    shifts = rng.choice(np.arange(1, N), size=min(50, N - 1), replace=False)
    fl = [per_sentence(real_h, [bt_refs[(i + s) % N] for i in range(N)]) for s in shifts]
    store["Wrong-sentence control (floor)"] = (np.mean([f[0] for f in fl], axis=0),
                                               fl[0][1], np.mean([f[2] for f in fl], axis=0))

    B = cfg.REC_BOOTSTRAP
    ix = np.random.default_rng(1).integers(0, N, size=(B, N))

    def boot(ed, rl, ex):
        return 100 * ex[ix].mean(1), 100 * ed[ix].sum(1) / rl[ix].sum(1)

    boots = {k: boot(*v) for k, v in store.items()}
    for name, (ed, rl, ex) in store.items():
        acc_b, wer_b = boots[name]
        is_floor = name.startswith("Wrong")
        hyps = bt_hyps.get(name)
        rows.append({
            "condition": name, "n_sentences": N,
            "sentence_exact_pct": 100 * ex.mean(),
            "sentence_exact_ci_lo": np.percentile(acc_b, 2.5), "sentence_exact_ci_hi": np.percentile(acc_b, 97.5),
            "wer_pct": 100 * ed.sum() / rl.sum(),
            "wer_ci_lo": np.percentile(wer_b, 2.5), "wer_ci_hi": np.percentile(wer_b, 97.5),
            "bleu1": float("nan") if is_floor else bleu(bt_refs, hyps, 1),
            "bleu2": float("nan") if is_floor else bleu(bt_refs, hyps, 2),
        })
    summ = pd.DataFrame(rows)
    print(summ.to_string(index=False, float_format=lambda x: f"{x:.1f}"))
    summ.to_csv(os.path.join(cfg.CHECKPOINT_DIR, "backtrans_summary.csv"), index=False)

    ceiling = summ.loc[summ.condition == "Real test poses", "sentence_exact_pct"].iloc[0]
    print("")
    print(f"VALIDITY GATE: recogniser reads {ceiling:.1f}% of REAL test sentences exactly "
          f"(criterion >= 90%). " + ("OK." if ceiling >= 90 else
          "NOT MET: the recogniser is too weak for model comparisons to be informative."))

    print("")
    print("Paired bootstrap differences in WER (percentage points; negative = first reads better):")
    pairs = [("NAT + MSE", "Progressive Transformer"),
             ("Diffusion (CFG 2.5)", "NAT + MSE"),
             ("Diffusion (CFG 2.5)", "Progressive Transformer"),
             ("Diffusion (CFG 4)", "Diffusion (CFG 1)")]
    for a, b in pairs:
        if a in boots and b in boots:
            d = boots[a][1] - boots[b][1]
            p = 2 * min((d <= 0).mean(), (d >= 0).mean())
            print(f"  {a} - {b}: {d.mean():+.1f} pp  95% CI [{np.percentile(d, 2.5):+.1f}, "
                  f"{np.percentile(d, 97.5):+.1f}]  p={p:.4f}")

    out = []
    for name, hyps in bt_hyps.items():
        ed, rl, ex = store[name]
        for i in range(N):
            out.append({"condition": name, "idx": i, "ref": " ".join(bt_refs[i]),
                        "hyp": " ".join(hyps[i]), "edits": int(ed[i]), "ref_len": int(rl[i]),
                        "exact": int(ex[i])})
    pd.DataFrame(out).to_csv(os.path.join(cfg.CHECKPOINT_DIR, "backtrans_per_sentence.csv"),
                             index=False, encoding="utf-8")
    print("saved backtrans_summary.csv and backtrans_per_sentence.csv")


# **16c. Three-Way Architecture Comparison**

Scores all three architectures (diffusion generator, NAT + MSE, Progressive Transformer;
Sections 9b, 14, 14b, 14c) with the identical `pose_metrics` / `evaluate_generation` pipeline on the
same held-out test set. The three share the data split, text-encoder architecture, batch
composition, epoch budget and early-stopping rule, so the numbers are directly comparable. The
table is reported as measured; no ordering is assumed in advance.


In [ ]:
import time as _time
import pandas as pd

print("Loading best checkpoints for all three architectures...")

_ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
text_encoder.load_state_dict(_ckpt['text_encoder_state_dict'])
duration_predictor.load_state_dict(_ckpt['duration_state_dict'])
diffusion_g2p.load_state_dict(_ckpt['diffusion_state_dict'])

_ckpt_prog = torch.load(PROG_BEST_PATH, map_location=device, weights_only=False)
text_encoder_prog.load_state_dict(_ckpt_prog['text_encoder_state_dict'])
progressive_decoder.load_state_dict(_ckpt_prog['decoder_state_dict'])

_ckpt_nat = torch.load(NAT_MSE_BEST_PATH, map_location=device, weights_only=False)
text_encoder_nat_mse.load_state_dict(_ckpt_nat['text_encoder_state_dict'])
duration_predictor_nat_mse.load_state_dict(_ckpt_nat['duration_state_dict'])
nat_mse_head.load_state_dict(_ckpt_nat['head_state_dict'])

_gen_diffusion = lambda ii, am, sid: generate_poses(
    ii, am, sid, text_encoder, duration_predictor, length_regulator,
    temporal_pos_encoder, diffusion_g2p, pose_refiner)
_gen_nat_mse = lambda ii, am, sid: generate_poses_nat_mse(ii, am, sid)
_gen_progressive = lambda ii, am, sid: generate_poses_progressive(ii, am, sid)

ARCH_VARIANTS = [
    ("Progressive Transformer", _gen_progressive, "Sequential (Slow)"),
    ("NAT + MSE", _gen_nat_mse, "Parallel (Fast)"),
    ("NAT + Diffusion + Velocity", _gen_diffusion, "Parallel (Fast)"),
]

comparison_rows = []
for name, gen_fn, paradigm in ARCH_VARIANTS:
    t0 = _time.time()
    recs = evaluate_generation(test_loader, gen_fn, desc=name)
    wall = _time.time() - t0
    m = summarize_metrics(recs)
    print_metrics(name, m, len(recs))
    comparison_rows.append({
        'model': name,
        'motion_ratio': m.get('motion_ratio', float('nan')),
        'dtw_mje': m.get('dtw_mje', float('nan')),
        'pck_0.05': m.get('PCK@0.05', float('nan')),
        'pck_0.1': m.get('PCK@0.1', float('nan')),
        'pck_0.2': m.get('PCK@0.2', float('nan')),
        'pck_0.5': m.get('PCK@0.5', float('nan')),
        'length_ratio': m.get('length_ratio', float('nan')),
        'inference_paradigm': paradigm,
        'test_wall_seconds': round(wall, 1),
        'n_test_videos': len(recs),
    })

    tag = name.split(':')[0].strip().lower().replace(' ', '_')
    path = os.path.join(cfg.CHECKPOINT_DIR, f"test_results_{tag}_seed{cfg.SEED}.csv")
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(recs[0].keys()))
        w.writeheader(); w.writerows(recs)
    print(f"Per-sample metrics -> {path}")

comparison_df = pd.DataFrame(comparison_rows)
print("\n" + "=" * 100)
print(" ARCHITECTURE COMPARISON — same split, same encoder family, same epoch/early-stop budget")
print("=" * 100)
print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

cmp_path = os.path.join(cfg.CHECKPOINT_DIR, "architecture_comparison.csv")
comparison_df.to_csv(cmp_path, index=False)
print(f"\nSaved -> {cmp_path}")


# **16d. Comparison Figure**

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
names_short = [{'Progressive Transformer': 'Progressive', 'NAT + MSE': 'NAT+MSE',
                'NAT + Diffusion + Velocity': 'Diffusion'}.get(r['model'], r['model'])
              for r in comparison_rows]
colors = ['#d62728', '#ff7f0e', '#2ca02c']

ax[0].bar(names_short, [r['motion_ratio'] for r in comparison_rows], color=colors)
ax[0].axhline(1.0, ls='--', c='gray', lw=1)
ax[0].set_title('Motion ratio (1.0 = matches GT)'); ax[0].set_ylabel('ratio')

ax[1].bar(names_short, [r['dtw_mje'] for r in comparison_rows], color=colors)
ax[1].set_title('DTW-MJE (lower is better)'); ax[1].set_ylabel('shoulder-widths')

ax[2].bar(names_short, [r['pck_0.5'] for r in comparison_rows], color=colors)
ax[2].set_title('PCK@0.5 (higher is better)'); ax[2].set_ylabel('%')

plt.suptitle('Architecture comparison on the held-out test set')
plt.tight_layout()
plt.show()


# **17. Visualization**

Poses are denormalized back into the original coordinate space before plotting,
so the skeleton has realistic proportions rather than shoulder-width units.

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

batch = next(iter(test_loader))
input_ids, attention_mask, target_poses, lengths, signer_ids, centers, scales = [
    b.to(device) for b in batch
]
gen, gen_lens = generate_poses(
    input_ids, attention_mask, signer_ids, text_encoder, duration_predictor,
    length_regulator, temporal_pos_encoder, diffusion_g2p, pose_refiner
)

idx = 0
seq = gen[idx, :int(gen_lens[idx].item())]
if cfg.NORMALIZE_POSES:  # invert the per-sequence normalization
    seq = seq * scales[idx] + centers[idx].view(1, 1, 3)
seq = seq.float().cpu().numpy()
num_frames = seq.shape[0]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
lo = seq.reshape(-1, 3).min(axis=0)
hi = seq.reshape(-1, 3).max(axis=0)
pad = 0.1 * (hi - lo + 1e-6)


def update_graph(frame):
    ax.clear()
    ax.set_title(f"Generated BdSL — signer {signer_ids[idx].item()} — frame {frame}/{num_frames}")
    ax.view_init(elev=10, azim=-90)
    ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0])
    ax.set_ylim(lo[2] - pad[2], hi[2] + pad[2])
    ax.set_zlim(-(hi[1] + pad[1]), -(lo[1] - pad[1]))
    p = seq[frame]
    ax.scatter(p[:33, 0], p[:33, 2], -p[:33, 1], c='blue', s=25)
    ax.scatter(p[33:75, 0], p[33:75, 2], -p[33:75, 1], c='green', s=15)
    for a, b in cfg.BONE_PAIRS:
        ax.plot([p[a, 0], p[b, 0]], [p[a, 2], p[b, 2]], [-p[a, 1], -p[b, 1]],
                c='red', lw=2)


print(f"Rendering {num_frames} frames...")
ani = animation.FuncAnimation(fig, update_graph, frames=num_frames, interval=50)
plt.close(fig)
HTML(ani.to_jshtml())


# **18. Pose-to-Video (Skeleton Overlay)**

Rule-based OpenCV skeleton rendering — not a learned photorealistic renderer.
This is not comparable to learned photorealistic video generation (for example SignGen or Text2Sign).

In [ ]:
import cv2


def project_to_pixel(x, y, x_range, y_range, w=256, h=256):
    px = int((x - x_range[0]) / (x_range[1] - x_range[0] + 1e-8) * w)
    py = int((y - y_range[0]) / (y_range[1] - y_range[0] + 1e-8) * h)
    return max(0, min(w - 1, px)), max(0, min(h - 1, py))


def synthesize_sign_video(pose_sequence, reference_image_path,
                          output_filename="bdsl_synthesis.mp4", fps=25):
    ref = cv2.imread(reference_image_path)
    if ref is None:
        print("Reference image not found — using blank canvas.")
        ref = np.ones((256, 256, 3), dtype=np.uint8) * 40
    else:
        ref = cv2.resize(ref, (256, 256))

    pose_np = pose_sequence.detach().float().cpu().numpy() \
        if torch.is_tensor(pose_sequence) else pose_sequence
    x_range = (float(pose_np[:, :, 0].min()), float(pose_np[:, :, 0].max()))
    y_range = (float(pose_np[:, :, 1].min()), float(pose_np[:, :, 1].max()))

    vw = cv2.VideoWriter(output_filename, cv2.VideoWriter_fourcc(*'mp4v'),
                         fps, (256, 256))
    for t in tqdm(range(pose_np.shape[0]), desc="Rendering video"):
        frame = ref.copy()
        p = pose_np[t]
        for a, b in cfg.BONE_PAIRS:
            p1 = project_to_pixel(p[a, 0], p[a, 1], x_range, y_range)
            p2 = project_to_pixel(p[b, 0], p[b, 1], x_range, y_range)
            cv2.line(frame, p1, p2, (0, 0, 255), 2)
        for j in range(cfg.HAND_END):
            px, py = project_to_pixel(p[j, 0], p[j, 1], x_range, y_range)
            cv2.circle(frame, (px, py), 3,
                       (0, 255, 0) if j >= cfg.HAND_START else (255, 0, 0), -1)
        vw.write(frame)
    vw.release()
    print(f"Saved {output_filename}")


den = gen[idx, :int(gen_lens[idx].item())]
if cfg.NORMALIZE_POSES:
    den = den * scales[idx] + centers[idx].view(1, 1, 3)
synthesize_sign_video(den, cfg.REF_IMAGE_PATH)


# **18b. Preserving Outputs (read before the session ends)**

In [ ]:
import os, shutil, glob

CK = cfg.CHECKPOINT_DIR
print("Files currently in", CK)
for f in sorted(glob.glob(os.path.join(CK, "*"))):
    print(f"  {os.path.getsize(f)/1e6:9.1f} MB  {os.path.basename(f)}")

# Bundle everything small and irreplaceable: metrics, history, config, figures,
# including the architecture-comparison outputs from Sections 15b/16c/16d.
# Checkpoints are excluded deliberately - they are large, and a run is
# reproducible from the seed, whereas per-sample results are not.
BUNDLE = os.path.join(CK, "run_artifacts")
os.makedirs(BUNDLE, exist_ok=True)
for pat in ("*.csv", "*.json", "*.mp4", "*.png"):
    for f in glob.glob(os.path.join(CK, pat)):
        shutil.copy2(f, BUNDLE)

with open(os.path.join(BUNDLE, "config_used.txt"), "w") as f:
    for k, v in sorted(vars(Config).items()):
        if not k.startswith("__"):
            f.write(f"{k} = {v}\n")

shutil.make_archive(os.path.join(CK, "run_artifacts"), "zip", BUNDLE)
print(f"\nWrote {CK}/run_artifacts.zip  "
     f"({os.path.getsize(os.path.join(CK,'run_artifacts.zip'))/1e6:.1f} MB)")
print("""
Download it NOW from the Output panel on the right.

To keep a checkpoint across sessions, either:
  1. Save Version -> Save & Run All, and the .pt files attach to that version, or
  2. Output panel -> Create Dataset, which makes them an input to future
     notebooks (this is the option that survives everything), or
  3. download g2p_best.pt / g2p_progressive_best.pt / g2p_nat_mse_best.pt
     directly (about 0.5 GB each).

*_best.pt holds weights only and is all evaluation needs.
*_last.pt additionally holds optimizer state and is only needed to RESUME.
""")


# **19. Reproducing the experiments**

This notebook is generated by `scripts/build_notebook.py`. Experiment variants (velocity and
guidance ablation, test-time smoothness guidance, NAT temporal-variance penalty, back-translation)
are built by `scripts/build_notebooks.py` and are documented, with their results, in the repository
README and `docs/`.

Practical notes:

- Develop at `SIGNERS_TO_USE = [0]` (about 2 h per full run on a T4); all five signers is roughly
  five times the data for the same vocabulary.
- Aggregate per sentence before any significance test when several signers record the same
  sentence: their recordings are correlated, so treating every video as independent overstates
  significance.
- Seeds: repeat the main configuration over several `SEED` values and report mean and spread.
- Human evaluation with BdSL signers is the measurement the automatic metrics here cannot replace.
